# 0: Imports, config

In [ ]:
from pathlib import Path
from typing import Tuple, Dict, List, Optional
import joblib
import os
import subprocess
import time
import json                                               #.. stages hand off through files                                              #.. per-model timing in the benchmark
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
try:
    from sklearn.exceptions import DataConversionWarning
    warnings.filterwarnings('ignore', category=DataConversionWarning)   #.. KNN jaccard spam
except ImportError:
    pass

from rdkit import Chem, RDLogger
from rdkit.Chem import (
    Descriptors, Lipinski, rdFingerprintGenerator,
    MolFromSmiles, FilterCatalog, AllChem, QED,         #.. QED for drug-likeness column
    DataStructs,                                        #.. for Tanimoto (AD + baseline model)
)
from rdkit.Chem.Scaffolds import MurckoScaffold         #.. scaffold split
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.inchi import MolToInchiKey              #.. duplicate detection
from rdkit.Avalon import pyAvalonTools                  #.. Avalon fingerprint
from rdkit.ML.Cluster import Butina                     #.. Butina cluster split
from rdkit.ML.Scoring import Scoring                    #.. BEDROC / enrichment factor
from rdkit.Chem import MACCSkeys                      
RDLogger.DisableLog("rdApp.*")          

# Model families for benchmarking                       
from sklearn.ensemble import (
    RandomForestClassifier, 
    GradientBoostingClassifier, 
    ExtraTreesClassifier, 
    AdaBoostClassifier, 
    HistGradientBoostingClassifier
)
from sklearn.svm import SVC, NuSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.base import BaseEstimator, ClassifierMixin  #.. for the Tanimoto baseline estimator
from sklearn.pipeline import Pipeline                    #.. scaling must be fit inside each fold
from sklearn.preprocessing import StandardScaler         #.. needed for rdkit_desc representation
from sklearn.model_selection import (
    RandomizedSearchCV,                                  #.. replaces GridSearchCV
    StratifiedKFold,                                     #.. explicit fold control
    RepeatedStratifiedKFold,                             #.. repeated CV for seed-variance reporting
    StratifiedGroupKFold,                                #.. grouped CV for scaffold/butina regimes
    cross_val_predict, cross_validate, train_test_split
)
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, matthews_corrcoef, f1_score, make_scorer,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    classification_report, cohen_kappa_score
)
from sklearn.base import clone
from sklearn.decomposition import PCA
from scipy.stats import mannwhitneyu, wilcoxon, loguniform, randint, uniform  #.. paired test + search distributions



try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True                                      #.. optional; skipped cleanly if absent
except ImportError:
    HAS_LGBM = False

## THROWS ERROR CATBOOST, numpy and catboost clash
# try:
#     from catboost import CatBoostClassifier
#     HAS_CAT = True
# except ImportError:
HAS_CAT = False



In [ ]:
class TargetConfig:
    """
    Single source of truth for paths and parameters.

    Every stage reads files and writes files; no stage depends on a variable
    left in memory by another stage. That is why bundle_path() is a function of
    the fingerprint name rather than one MODEL_PATH.
    """

    def __init__(self, target_name, target_id, base_dir=None, seed=42,
                 active_pchembl=7.0, inactive_pchembl=7.0, drop_grey_zone=True,
                 fp_radius=2, fp_bits=2048, domain="BD1"):
        self.PROTEIN_NAME = target_name
        self.PROTEIN_ID   = target_id

        if base_dir is None:
            base_dir = Path.cwd().parent / "data"
        self.BASE_DIR   = Path(base_dir)
        self.TARGET_DIR = self.BASE_DIR / target_name

        # --- inputs ---
        self.DATA_PATH    = self.TARGET_DIR / "ml" / f"{target_id}_raw.csv"
        self.COCONUT_CSV  = self.BASE_DIR / "coconut" / "coconut_csv-09-2026.csv"
        self.COCONUT_SDF  = self.BASE_DIR / "coconut" / "coconut_sdf_3d-09-2026.sdf"
        self.INPUT_CSV    = self.COCONUT_CSV

        # --- directories ---
        self.ML_DIR      = self.TARGET_DIR / "ml"
        self.BUNDLE_DIR  = self.ML_DIR / "bundles"          #.. one .pkl per representation
        self.FIGURE_DIR  = self.TARGET_DIR / "out" / "figures"
        self.RESULTS_DIR = self.TARGET_DIR / "out" / "results"
        self.DOCKING_DIR = self.TARGET_DIR / "docking"
        for d in (self.ML_DIR, self.BUNDLE_DIR, self.FIGURE_DIR,
                  self.RESULTS_DIR, self.DOCKING_DIR):
            d.mkdir(parents=True, exist_ok=True)

        # --- stage outputs (files, not variables) ---
        tag = f"{target_id}_{domain}"
        self.CURATED_PATH       = self.ML_DIR / f"{tag}_curated.csv"
        self.HITS_PATH          = self.TARGET_DIR / f"{tag}_coconut_scored.csv"
        self.FINAL_DOCKING_LIST = self.TARGET_DIR / f"{tag}_final_docking_list.csv"
        self.CONTROLS_PATH      = self.TARGET_DIR / f"{tag}_random_controls.csv"

        # --- parameters ---
        self.SEED             = seed
        self.ACTIVE_PCHEMBL   = active_pchembl
        self.INACTIVE_PCHEMBL = inactive_pchembl
        self.DROP_GREY_ZONE   = drop_grey_zone
        self.DOMAIN           = domain
        self.FP_RADIUS        = fp_radius
        self.FP_BITS          = fp_bits

        # --- metric configuration -------------------------------------------
        # Two separate fraction sets, because the same cut means different
        # things at different N. On a ~790-compound test set, 1% is 8 molecules
        # and nEF saturates at 1.000; on a 695k library, 1% is ~7,000.
        self.EF_FRACTIONS_CV     = [0.05, 0.10]     #.. cross-validation / held-out test
        self.EF_FRACTIONS_SCREEN = [0.005, 0.01]    #.. COCONUT screen reporting
        self.BEDROC_ALPHA        = 20.0
        self.RANKING_METRIC      = "BEDROC"         #.. single selection metric

        # --- screening ---
        self.N_DOCK       = 150
        self.PER_MODEL_N  = 60                      #.. 2 reps x 60 = 120, round-robin to 150
        self.AD_TC_BANDS  = [0.3, 0.5]
        self.N_CONTROLS   = 200

    def bundle_path(self, fp_name):
        """One self-contained model artifact per representation."""
        return self.BUNDLE_DIR / f"{self.PROTEIN_ID}_{self.DOMAIN}_{fp_name}.pkl"

    def result_path(self, name, ext="json"):
        return self.RESULTS_DIR / f"{name}.{ext}"

In [ ]:
# ========= Compute profile =========
COMPUTE_PROFILE = "college"          # "mac" | "college"

# All profiles carry the SAME keys, so no function can KeyError on one machine
# and work on another.
_PROFILES = {
    "mac":     dict(N_JOBS=6,  SCREEN_ITER=0, TUNE_ITER=20, N_REPEATS=1,
                    N_SPLITS=5, Y_SCRAMBLE=10, SEED_SENS=10),
    "college": dict(N_JOBS=-1, SCREEN_ITER=30, TUNE_ITER=60, N_REPEATS=10,
                    N_SPLITS=5, Y_SCRAMBLE=20, SEED_SENS=10),
}
BUDGET = _PROFILES[COMPUTE_PROFILE]
print(f"compute profile: {COMPUTE_PROFILE} -> {BUDGET}")

# ========= Experiment =========
FP_BITS          = 2048
ACTIVE_PCHEMBL   = 7.0     # IC50 <= 100 nM
INACTIVE_PCHEMBL = 7.0     # equal to ACTIVE  ->  single cut, no grey zone
DROP_GREY_ZONE   = True
DOMAIN           = "BD1"

cfg = TargetConfig(
    target_name="brd4", target_id="CHEMBL1163125",
    fp_bits=FP_BITS, active_pchembl=ACTIVE_PCHEMBL,
    inactive_pchembl=INACTIVE_PCHEMBL, drop_grey_zone=DROP_GREY_ZONE,
    domain=DOMAIN,
)

# Representations carried through the pipeline. Chosen for FAMILY DISTINCTNESS
# (consensus only helps when methods differ), not by ranking: the fingerprint
# screen showed all candidates within noise of each other.
PRIMARY_FP  = "ecfp4_maccs"     # hybrid, Ahmad et al. 2026
SECONDARY_FP = "avalon"         # algorithmically distinct from circular + keys
TOP_FPS = [PRIMARY_FP, SECONDARY_FP]



# NOTE: ScreeningMetrics.configure(cfg) is called at the top of STAGE 1,
# because the class is defined in a later cell than this one.

In [ ]:
SMOKE = False                    #.. one hyperparameter config only; MUST be False for the real run
if SMOKE:
    BUDGET = dict(BUDGET)       # don't mutate the shared profile dict
    BUDGET.update(TUNE_ITER=1, SCREEN_ITER=0, N_REPEATS=1,
                  Y_SCRAMBLE=3, SEED_SENS=3)
    print("*** SMOKE MODE: results are NOT publishable ***")



if SMOKE:
    cfg.RESULTS_DIR = cfg.RESULTS_DIR.parent / "results_smoke"      #.. separate directory
    cfg.FIGURE_DIR  = cfg.FIGURE_DIR.parent / "figures_smoke"
    cfg.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    cfg.FIGURE_DIR.mkdir(parents=True, exist_ok=True)


# 1. Machine Learning

## 1.1 Background Source Code

In [ ]:
class ChemStandardizer:

    """

    Structure standardization and duplicate detection.



    Runs the RDKit MolStandardize pipeline (cleanup -> largest fragment ->

    uncharge -> canonical tautomer) so that salts, charge states and tautomers

    of the same compound collapse to one record BEFORE any train/test split.

    Without this, the same molecule can appear on both sides of a split under

    two different SMILES strings, which inflates test performance.

    """



    _lfc = None

    _uncharger = None

    _taut = None



    @classmethod

    def _init(cls):

        if cls._lfc is None:

            cls._lfc = rdMolStandardize.LargestFragmentChooser()

            cls._uncharger = rdMolStandardize.Uncharger()

            p = rdMolStandardize.CleanupParameters()

            p.maxTautomers = 100

            cls._taut = rdMolStandardize.TautomerEnumerator(p)



    @staticmethod

    def standardize_mol(smiles, do_tautomer=True):

        """Returns a standardized RDKit Mol, or None if the SMILES is unusable."""

        ChemStandardizer._init()

        mol = Chem.MolFromSmiles(str(smiles))

        if mol is None:

            return None

        try:

            mol = rdMolStandardize.Cleanup(mol)

            mol = ChemStandardizer._lfc.choose(mol)      # strip salts / counterions

            mol = ChemStandardizer._uncharger.uncharge(mol)

            if do_tautomer:

                mol = ChemStandardizer._taut.Canonicalize(mol)

            Chem.SanitizeMol(mol)

        except Exception:

            return None

        if mol is None or mol.GetNumHeavyAtoms() == 0:

            return None

        return mol



    @staticmethod

    def standardize_frame(df, smiles_col="canonical_smiles", do_tautomer=True, verbose=True):

        """

        Adds std_smiles, inchikey and inchikey_skeleton columns.



        inchikey       : full key, identifies an exact structure incl. stereochemistry

        inchikey_skeleton : first block only, so stereoisomers of one compound match

        """

        rows, n_fail = [], 0

        for smi in df[smiles_col].astype(str):

            mol = ChemStandardizer.standardize_mol(smi, do_tautomer)

            if mol is None:

                n_fail += 1

                rows.append((None, None, None))

                continue

            try:

                ik = MolToInchiKey(mol)

            except Exception:

                ik = None

            rows.append((Chem.MolToSmiles(mol), ik, ik.split("-")[0] if ik else None))



        out = df.copy().reset_index(drop=True)

        std = pd.DataFrame(rows, columns=["std_smiles", "inchikey", "inchikey_skeleton"])

        out = pd.concat([out, std], axis=1)

        before = len(out)

        out = out.dropna(subset=["std_smiles", "inchikey"]).reset_index(drop=True)

        if verbose:

            print(f"  standardization: {before} in -> {len(out)} out ({n_fail} failed to parse)")

        return out



    @staticmethod

    def deduplicate(df, value_col="pchembl_value", agg="median", verbose=True):

        """

        Collapses records to one row per unique structure.



        Aggregates the activity by median rather than mean: ChEMBL replicate

        measurements are log-scale and skewed, so the median is less sensitive

        to a single bad assay.

        """

        n0 = len(df)

        n_exact = n0 - df["inchikey"].nunique()

        n_stereo = df["inchikey"].nunique() - df["inchikey_skeleton"].nunique()

        keep = {c: "first" for c in df.columns if c not in ("inchikey", value_col)}

        keep[value_col] = agg

        out = df.groupby("inchikey", as_index=False).agg(keep)

        if verbose:

            print(f"  dedup: {n0} -> {len(out)} rows | exact duplicates removed: {n_exact} "

                  f"| stereoisomer-only pairs remaining: {n_stereo}")

        return out



    @staticmethod

    def replicate_spread(df_raw, key="inchikey", value_col="pchembl_value",

                         lo=5.0, hi=6.0, verbose=True):

        """

        Reports how many compounds have replicate measurements that straddle the

        activity cutoffs. This is the empirical justification for the grey zone

        and belongs in the methods section.

        """

        g = df_raw.groupby(key)[value_col]

        n_rep = (g.count() > 1).sum()

        spread = (g.max() - g.min())

        straddle = ((g.min() < lo) & (g.max() >= hi)).sum()

        if verbose:

            print(f"  replicates: {n_rep} compounds measured more than once")

            print(f"  median replicate spread: {spread[spread > 0].median():.2f} log units")

            print(f"  compounds whose replicates straddle {lo}-{hi}: {straddle}")

        return dict(n_replicated=int(n_rep),

                    median_spread=float(spread[spread > 0].median()) if (spread > 0).any() else 0.0,

                    n_straddle=int(straddle))

In [ ]:
class ChemUtils:

    """

    Static utilities for turning SMILES into features and property columns.



    Two behavioural changes from the thesis version:

      1. PAINS and Brenk are separate, return the matched alert NAMES, and

         distinguish an unparseable molecule from a flagged one.

      2. Fingerprint generators return (X, ok_mask) instead of padding failures

         with an all-zero row. A zero vector is a real feature vector to the

         model; padding silently injects phantom compounds into training.

         The caller MUST subset its dataframe with ok_mask.

    """



    # ---------- structural alerts ----------                      #.. was one merged boolean

    _pains = None

    _brenk = None



    @classmethod

    def _catalog(cls, which):

        if which == "PAINS":

            if cls._pains is None:

                p = FilterCatalog.FilterCatalogParams()

                p.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)

                cls._pains = FilterCatalog.FilterCatalog(p)

            return cls._pains

        if cls._brenk is None:

            p = FilterCatalog.FilterCatalogParams()

            p.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)

            cls._brenk = FilterCatalog.FilterCatalog(p)

        return cls._brenk



    @staticmethod

    def _alerts(smiles, which):

        """Returns (status, alert_names) where status is CLEAN | FLAG | INVALID."""

        mol = Chem.MolFromSmiles(str(smiles))

        if mol is None:

            return "INVALID", ""                                   #.. no longer conflated with "bad"

        matches = ChemUtils._catalog(which).GetMatches(mol)

        if not matches:

            return "CLEAN", ""

        return "FLAG", ";".join(sorted({m.GetDescription() for m in matches}))



    @staticmethod

    def check_pains(smiles):

        return ChemUtils._alerts(smiles, "PAINS")



    @staticmethod

    def check_brenk(smiles):

        return ChemUtils._alerts(smiles, "BRENK")



    @staticmethod

    def annotate_alerts(df, smiles_col="std_smiles"):

        """

        Adds pains_status / pains_alerts / brenk_status / brenk_alerts as COLUMNS.

        Nothing is dropped. Filtering here would remove the catechol and quinone

        natural products that are the most plausible bromodomain chemotypes.

        """

        out = df.copy()

        p = out[smiles_col].apply(lambda s: pd.Series(ChemUtils.check_pains(s)))

        b = out[smiles_col].apply(lambda s: pd.Series(ChemUtils.check_brenk(s)))

        out["pains_status"], out["pains_alerts"] = p[0], p[1]

        out["brenk_status"], out["brenk_alerts"] = b[0], b[1]

        return out



    # ---------- fingerprint generators ----------                 #.. was one generate_fingerprints()

    _gens = {}



    @classmethod

    def _morgan(cls, radius, bits, chirality):

        k = ("m", radius, bits, chirality)

        if k not in cls._gens:

            cls._gens[k] = rdFingerprintGenerator.GetMorganGenerator(

                radius=radius, fpSize=bits, includeChirality=chirality)

        return cls._gens[k]



    @classmethod

    def _rdk(cls, maxpath, bits):

        k = ("r", maxpath, bits)

        if k not in cls._gens:

            cls._gens[k] = rdFingerprintGenerator.GetRDKitFPGenerator(maxPath=maxpath, fpSize=bits)

        return cls._gens[k]



    @staticmethod

    def _run(smiles_list, fn, width):

        """Applies fn to each molecule. Returns (X, ok_mask); failures are EXCLUDED."""

        rows, ok = [], []

        for s in smiles_list:

            mol = Chem.MolFromSmiles(str(s))

            if mol is None:

                ok.append(False); continue

            try:

                rows.append(fn(mol)); ok.append(True)

            except Exception:

                ok.append(False)

        ok = np.array(ok, dtype=bool)

        X = np.array(rows, dtype=np.float32) if rows else np.zeros((0, width), dtype=np.float32)

        return X, ok



    @staticmethod

    def generate_ecfp4(smiles_list, bits=2048, counts=False, chirality=True):

        """Morgan radius 2. chirality=True matters for COCONUT, which is stereo-rich."""

        g = ChemUtils._morgan(2, bits, chirality)

        return ChemUtils._run(smiles_list,

                              g.GetCountFingerprintAsNumPy if counts else g.GetFingerprintAsNumPy,

                              bits)



    @staticmethod

    def generate_ecfp6(smiles_list, bits=2048, counts=False, chirality=True):

        """Morgan radius 3."""

        g = ChemUtils._morgan(3, bits, chirality)

        return ChemUtils._run(smiles_list,

                              g.GetCountFingerprintAsNumPy if counts else g.GetFingerprintAsNumPy,

                              bits)



    @staticmethod

    def generate_rdk7(smiles_list, bits=2048, counts=False):

        """RDKit path-based fingerprint, maxPath 7. Algorithmically distinct from Morgan."""

        g = ChemUtils._rdk(7, bits)

        return ChemUtils._run(smiles_list,

                              g.GetCountFingerprintAsNumPy if counts else g.GetFingerprintAsNumPy,

                              bits)



    @staticmethod

    def generate_maccs(smiles_list, bits=None, counts=False):

        """

        166 predefined structural keys (RDKit returns 167; bit 0 is unused).



        Kept as a distinct family because it is the only representation whose

        features are human-readable, which makes a feature-importance table

        possible, and because generic predefined keys transfer to out-of-domain

        natural-product scaffolds better than hashed circular environments.

        """

        return ChemUtils._run(smiles_list,

                              lambda m: np.array(MACCSkeys.GenMACCSKeys(m)), 167)





    @staticmethod

    def generate_avalon(smiles_list, bits=2048, counts=False):

        """Avalon fingerprint."""

        if counts:

            f = lambda m: np.array(pyAvalonTools.GetAvalonCountFP(m, nBits=bits).ToList())

        else:

            f = lambda m: np.array(pyAvalonTools.GetAvalonFP(m, nBits=bits))

        return ChemUtils._run(smiles_list, f, bits)





    @staticmethod

    def generate_hybrid(smiles_list, parts, bits=2048):

        """

        Column-wise concatenation of two or more representations.



        The alignment matters: each generator returns only its OK rows, so the

        blocks cannot be hstacked directly. Each is scattered back into a

        full-length array first, then the intersection of the OK masks is taken,

        so row i of the output always corresponds to the same molecule in every

        block.

        """

        smiles_list = list(smiles_list)

        n = len(smiles_list)

        blocks, ok_all = [], np.ones(n, dtype=bool)

        for name in parts:

            Xp, okp = ChemUtils.FP_REGISTRY[name](smiles_list, bits)

            full = np.zeros((n, Xp.shape[1]), dtype=np.float32)

            full[okp] = Xp                      # scatter back to full length

            blocks.append(full)

            ok_all &= okp                       # a molecule must succeed in ALL parts

        return np.hstack(blocks)[ok_all], ok_all



    _DESC = None



    @classmethod

    def _desc_list(cls):

        if cls._DESC is None:

            # Ipc is excluded: it overflows to inf on larger natural products

            cls._DESC = [(n, f) for n, f in Descriptors._descList if n != "Ipc"]

        return cls._DESC



    @staticmethod

    def generate_rdkit_desc(smiles_list, **kw):

        """

        RDKit 2D descriptor block (~215 columns). This is the out-of-domain hedge:

        when a COCONUT scaffold shares no substructure bits with the ChEMBL training

        set, descriptor models degrade more gracefully than fingerprint models.

        MUST be used inside a Pipeline with StandardScaler.

        """

        dl = ChemUtils._desc_list()

        def f(mol):

            v = []

            for _, fn in dl:

                try:

                    v.append(float(fn(mol)))

                except Exception:

                    v.append(0.0)

            return v

        X, ok = ChemUtils._run(smiles_list, f, len(dl))

        if X.size:

            X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

        return X, ok



    # ---------- dispatcher used by the fingerprint screen ----------   #.. new

    FP_REGISTRY = {

        "ecfp4":       lambda s, b: ChemUtils.generate_ecfp4(s, b, counts=False),

        "ecfp6":       lambda s, b: ChemUtils.generate_ecfp6(s, b, counts=False),

        "ecfp4_count": lambda s, b: ChemUtils.generate_ecfp4(s, b, counts=True),

        "ecfp6_count": lambda s, b: ChemUtils.generate_ecfp6(s, b, counts=True),

        "avalon":      lambda s, b: ChemUtils.generate_avalon(s, b, counts=False),

        "rdk7":        lambda s, b: ChemUtils.generate_rdk7(s, b, counts=False),

        "rdkit_desc":  lambda s, b: ChemUtils.generate_rdkit_desc(s),

        "maccs": lambda s, b: ChemUtils.generate_maccs(s),

        "ecfp4_maccs": lambda s, b: ChemUtils.generate_hybrid(s, ["ecfp4", "maccs"], b), #.. Ahmad et al. 2026 hybrid

        "ecfp6_maccs": lambda s, b: ChemUtils.generate_hybrid(s, ["ecfp6", "maccs"], b), #..

    }

    # binary representations only: Tanimoto kernel / NN-Tanimoto / BernoulliNB need bit vectors

    BINARY_FPS = {"ecfp4", "ecfp6", "avalon", "rdk7", "maccs", "ecfp4_maccs", "ecfp6_maccs"}

    NEEDS_SCALING = {"rdkit_desc"}



    @staticmethod

    def get_fingerprint(name, smiles_list, bits=2048):

        """Returns (X, ok_mask). Subset your dataframe with ok_mask before using X."""

        if name not in ChemUtils.FP_REGISTRY:

            raise KeyError(f"unknown fingerprint '{name}'. options: {list(ChemUtils.FP_REGISTRY)}")

        return ChemUtils.FP_REGISTRY[name](list(smiles_list), bits)



    # ---------- property columns ----------                         #.. merges the two old methods

    @staticmethod

    def calculate_descriptors(smiles):

        """

        Physicochemical columns for EDA, reporting and the docking-stage size limits.

        RO5_violations is REPORTED, never used to drop rows.

        """

        idx = ["MW","LogP","HBD","HBA","RotBonds","TPSA",

               "HeavyAtoms","RingCount","FracCSP3","QED","RO5_violations"]

        mol = Chem.MolFromSmiles(str(smiles))

        if mol is None:

            return pd.Series([np.nan]*len(idx), index=idx)

        mw, logp = Descriptors.MolWt(mol), Descriptors.MolLogP(mol)

        hbd, hba = Lipinski.NumHDonors(mol), Lipinski.NumHAcceptors(mol)

        viol = int(mw > 500) + int(logp > 5) + int(hbd > 5) + int(hba > 10)

        return pd.Series([mw, logp, hbd, hba,

                          Descriptors.NumRotatableBonds(mol), Descriptors.TPSA(mol),

                          mol.GetNumHeavyAtoms(), Descriptors.RingCount(mol),

                          Descriptors.FractionCSP3(mol), QED.qed(mol), viol], index=idx)

In [ ]:
class SimilarityUtils:

    """

    Bulk Tanimoto over binary ECFP4. Used in three places, deliberately kept

    in one class so all three use the same definition:

      1. leakage audit  (test -> train similarity)

      2. applicability domain (COCONUT -> train similarity)

      3. the NN-Tanimoto baseline model (chunk C)



    Chirality is OFF here on purpose. For similarity we want two stereoisomers

    to look like the same compound, which is exactly the leakage we are hunting.

    """



    @staticmethod

    def to_bits(smiles_list, bits=2048):

        """Binary ECFP4 as float32. Returns (X, ok_mask)."""

        X, ok = ChemUtils.generate_ecfp4(smiles_list, bits=bits, counts=False, chirality=False)

        return (X > 0).astype(np.float32), ok



    @staticmethod

    def tanimoto_matrix(A, B, block=2000):

        """

        Exact Tanimoto for binary vectors via matrix product:

            Tc = |A n B| / (|A| + |B| - |A n B|)

        Blocked so a 700k x 5k comparison does not allocate one giant array.

        """

        A = A.astype(np.float32); B = B.astype(np.float32)

        a = A.sum(1)[:, None]; b = B.sum(1)[None, :]

        out = np.empty((A.shape[0], B.shape[0]), dtype=np.float32)

        for i in range(0, A.shape[0], block):

            j = min(i + block, A.shape[0])

            inter = A[i:j] @ B.T

            den = a[i:j] + b - inter

            out[i:j] = np.divide(inter, den, out=np.zeros_like(inter), where=den > 0)

        return out



    @staticmethod

    def max_tanimoto(query_bits, ref_bits, block=2000):

        """Max Tc of each query against the whole reference set. Never holds the full matrix."""

        best = np.zeros(query_bits.shape[0], dtype=np.float32)

        for i in range(0, query_bits.shape[0], block):

            j = min(i + block, query_bits.shape[0])

            best[i:j] = SimilarityUtils.tanimoto_matrix(query_bits[i:j], ref_bits).max(axis=1)

        return best





class Splitters:

    """

    Three split regimes, all returning (train_idx, test_idx, group_labels).



    Random is what the literature reports and what makes your numbers comparable.

    Scaffold and Butina are the honest estimates. Run all three on the SAME

    dataset: each is an independent 80/20 partition.



    The gap between random and Butina is your leakage estimate and is a headline

    result.

    """



    @staticmethod

    def murcko_scaffolds(smiles_list, include_chirality=False):

        """Bemis-Murcko scaffold SMILES per molecule. Acyclic molecules get __NOSCAFFOLD__."""

        out = []

        for s in smiles_list:

            try:

                sc = MurckoScaffold.MurckoScaffoldSmiles(smiles=str(s), includeChirality=include_chirality)

                out.append(sc if sc else "__NOSCAFFOLD__")

            except Exception:

                out.append("__NOSCAFFOLD__")

        return np.array(out, dtype=object)



    @staticmethod

    def butina_clusters(smiles_list, cutoff=0.65, bits=2048, verbose=True):

        """

        Butina clustering implemented directly on a numpy neighbour matrix rather

        than handing RDKit a flat distance list, which for n>5000 would be a

        multi-hundred-MB Python list.



        cutoff is a SIMILARITY threshold (0.65 = Tc 0.65), not a distance.

        Lower cutoff -> fewer, larger clusters -> harsher split.

        """

        bitsX, ok = SimilarityUtils.to_bits(smiles_list, bits)

        n = bitsX.shape[0]

        S = SimilarityUtils.tanimoto_matrix(bitsX, bitsX)

        nbr = S >= cutoff

        np.fill_diagonal(nbr, True)

        order = np.argsort(-nbr.sum(1))          # densest neighbourhoods become centroids first

        labels = np.full(n, -1, dtype=int)

        cid = 0

        for i in order:

            if labels[i] != -1:

                continue

            labels[nbr[i] & (labels == -1)] = cid

            cid += 1

        if verbose:

            sizes = np.bincount(labels)

            print(f"  butina(Tc>={cutoff}): {n} mols -> {cid} clusters "

                  f"(largest {sizes.max()}, singletons {(sizes == 1).sum()})")

        return labels, ok



    @staticmethod

    def _group_split(groups, y, test_size, seed, verbose=True, name="group"):

        """Assigns WHOLE groups to test, largest first, capped at 110% of target size."""

        rng = np.random.default_rng(seed)

        y = np.asarray(y)

        uniq = list(pd.unique(groups))

        rng.shuffle(uniq)

        uniq.sort(key=lambda g: -int(np.sum(groups == g)))

        n_target = int(round(test_size * len(y)))

        cap = int(n_target * 1.10)

        test_idx, n_test = [], 0

        for g in uniq:

            idx = np.where(groups == g)[0]

            if n_test + len(idx) <= cap:

                test_idx.extend(idx); n_test += len(idx)

            if n_test >= n_target:

                break

        if n_test < n_target * 0.5 and len(uniq):      # pathological case: one huge group

            biggest = np.where(groups == uniq[0])[0]

            test_idx = list(biggest)

        test_idx = np.array(sorted(test_idx), dtype=int)

        train_idx = np.setdiff1d(np.arange(len(y)), test_idx)

        if verbose:

            print(f"  {name} split: train {len(train_idx)} (act {y[train_idx].mean():.1%}) | "

                  f"test {len(test_idx)} (act {y[test_idx].mean():.1%}) "

                  f"[target n={n_target}, achieved {len(test_idx)}]")

            if abs(len(test_idx) - n_target) > 0.25 * n_target:

                print(f"    WARNING: group sizes prevent a clean {test_size:.0%} split "

                      f"({len(uniq)} groups).")

            if abs(y[test_idx].mean() - y.mean()) > 0.10:

                print(f"    WARNING: test active rate {y[test_idx].mean():.1%} vs overall "

                      f"{y.mean():.1%}. Report this; grouped splits cannot be fully stratified.")

        return train_idx, test_idx



    @staticmethod

    def random_split(y, test_size=0.2, seed=42, verbose=True):

        """Stratified random split. Returns (train_idx, test_idx, None)."""

        rng = np.random.default_rng(seed)

        y = np.asarray(y); idx = np.arange(len(y)); tr, te = [], []

        for c in np.unique(y):

            ci = idx[y == c]; rng.shuffle(ci)

            k = int(round(test_size * len(ci)))

            te.extend(ci[:k]); tr.extend(ci[k:])

        tr, te = np.array(sorted(tr)), np.array(sorted(te))

        if verbose:

            print(f"  random split: train {len(tr)} (act {y[tr].mean():.1%}) | "

                  f"test {len(te)} (act {y[te].mean():.1%})")

        return tr, te, None



    @staticmethod

    def scaffold_split(smiles_list, y, test_size=0.2, seed=42, verbose=True):

        """Bemis-Murcko split. Returns (train_idx, test_idx, scaffold_labels)."""

        g = Splitters.murcko_scaffolds(smiles_list)

        if verbose:

            print(f"  scaffolds: {len(pd.unique(g))} unique for {len(y)} compounds")

        tr, te = Splitters._group_split(g, y, test_size, seed, verbose, "scaffold")

        return tr, te, g



    @staticmethod

    def butina_split(smiles_list, y, test_size=0.2, seed=42, cutoff=0.65, verbose=True):

        """Cluster split. Returns (train_idx, test_idx, cluster_labels)."""

        labels, ok = Splitters.butina_clusters(smiles_list, cutoff, verbose=verbose)

        assert ok.all(), "butina_split: some SMILES failed to fingerprint; standardize upstream"

        tr, te = Splitters._group_split(labels, y, test_size, seed, verbose, "butina")

        return tr, te, labels



    @staticmethod

    def make_cv(n_splits=5, groups=None, seed=42):

        """

        CV object matching the split regime. If the outer split is grouped, the

        inner CV must be grouped too, or the CV estimate is optimistic even

        though the test set is clean.

        """

        if groups is None:

            return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)



    @staticmethod

    def leakage_audit(smiles_train, smiles_test, y_train=None, y_test=None,

                      inchikey_train=None, inchikey_test=None,

                      skeleton_train=None, skeleton_test=None,

                      tc_flag=0.6, verbose=True):

        """

        Quantifies four leakage categories:

          exact_overlap        identical structures on both sides (InChIKey)

          skeleton_overlap     stereoisomers of the same compound on both sides

          analog_active_pairs  active-active pairs at Tc >= tc_flag across the split

          max_tc distribution  the figure that goes in your EDA section



        Run for EVERY split regime. exact_overlap and skeleton_overlap must be 0.

        """

        rep = {}

        if inchikey_train is not None:

            rep["exact_overlap"] = len(set(inchikey_train) & set(inchikey_test))

        if skeleton_train is not None:

            rep["skeleton_overlap"] = len(set(skeleton_train) & set(skeleton_test))



        btr, ok_tr = SimilarityUtils.to_bits(smiles_train)

        bte, ok_te = SimilarityUtils.to_bits(smiles_test)

        maxtc = SimilarityUtils.max_tanimoto(bte, btr)

        rep["max_tc"] = maxtc

        rep["median_max_tc"] = float(np.median(maxtc))

        rep["frac_above_flag"] = float((maxtc >= tc_flag).mean())



        if y_train is not None and y_test is not None:

            ta = btr[np.asarray(y_train)[ok_tr] == 1]

            ea = bte[np.asarray(y_test)[ok_te] == 1]

            if len(ta) and len(ea):

                S = SimilarityUtils.tanimoto_matrix(ea, ta)

                rep["analog_active_pairs"] = int((S >= tc_flag).sum())

                rep["max_tc_active"] = S.max(axis=1)



        if verbose:

            print("  LEAKAGE AUDIT")

            for k in ("exact_overlap", "skeleton_overlap", "analog_active_pairs"):

                if k in rep:

                    print(f"    {k}: {rep[k]}")

            print(f"    median max Tc(test -> train): {rep['median_max_tc']:.3f}")

            print(f"    fraction of test with max Tc >= {tc_flag}: {rep['frac_above_flag']:.1%}")

        return rep

In [ ]:
class ScreeningMetrics:
    """
    Early-recognition metrics.

    CONFIGURATION: call ScreeningMetrics.configure(cfg) once, right after cfg is
    built. Every method then defaults to the configured alpha and fractions.
    Previously these were hardcoded defaults in five separate signatures, so
    changing cfg.EF_FRACTIONS silently changed nothing and then crashed the
    figures. Nothing in this class reads a literal fraction any more.
    """

    ALPHA = 20.0                    #.. set by configure()
    FRACTIONS = (0.05, 0.10)        #.. set by configure(); CV/test fractions
    FRACTIONS_SCREEN = (0.005, 0.01)

    @classmethod
    def configure(cls, cfg):
        cls.ALPHA = float(cfg.BEDROC_ALPHA)
        cls.FRACTIONS = tuple(cfg.EF_FRACTIONS_CV)
        cls.FRACTIONS_SCREEN = tuple(cfg.EF_FRACTIONS_SCREEN)
        return cls

    @classmethod
    def _f(cls, fractions):
        return cls.FRACTIONS if fractions is None else tuple(fractions)

    @classmethod
    def _a(cls, alpha):
        return cls.ALPHA if alpha is None else float(alpha)

    @staticmethod
    def _ranked(y_true, y_score):
        y_true = np.asarray(y_true).astype(int)
        y_score = np.asarray(y_score, dtype=float)
        order = np.argsort(-y_score, kind="mergesort")
        return [[int(v)] for v in y_true[order]]

    @classmethod
    def bedroc(cls, y_true, y_score, alpha=None):
        """alpha=20 puts ~80% of the weight in the top 8% of the ranking."""
        return float(Scoring.CalcBEDROC(ScreeningMetrics._ranked(y_true, y_score),
                                        0, cls._a(alpha)))

    @staticmethod
    def ef_max(y_true, fraction):
        """Theoretical max EF = min(N/n_actives, 1/fraction)."""
        y = np.asarray(y_true).astype(int)
        N, na = len(y), int(y.sum())
        return np.nan if na == 0 else float(min(N / na, 1.0 / fraction))

    @classmethod
    def enrichment(cls, y_true, y_score, fractions=None, normalize=True):
        fr = cls._f(fractions)
        ef = Scoring.CalcEnrichment(ScreeningMetrics._ranked(y_true, y_score), 0, list(fr))
        out = {}
        for f, e in zip(fr, ef):
            out[f"EF{f*100:g}%"] = float(e)
            if normalize:
                m = ScreeningMetrics.ef_max(y_true, f)
                out[f"nEF{f*100:g}%"] = float(e) / m if m and m > 0 else np.nan
        return out

    @classmethod
    def nef(cls, y_true, y_score, fraction=None):
        """Single nEF value, for use as an sklearn scorer."""
        f = cls.FRACTIONS[0] if fraction is None else fraction
        return cls.enrichment(y_true, y_score, (f,))[f"nEF{f*100:g}%"]

    @classmethod
    def primary_nef_key(cls):
        """Name of the nEF column used for ranking/plot titles. Never hardcode 'nEF1%'."""
        return f"nEF{cls.FRACTIONS[0]*100:g}%"

    @classmethod
    def calibration(cls, y_true, y_score, n_bins=10):
        """Brier score and Expected Calibration Error (Ahmad et al. 2026 report both)."""
        from sklearn.metrics import brier_score_loss
        y_true = np.asarray(y_true).astype(int); y_score = np.asarray(y_score, dtype=float)
        edges = np.quantile(y_score, np.linspace(0, 1, n_bins + 1))
        edges[0], edges[-1] = -np.inf, np.inf
        ece = 0.0
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (y_score > lo) & (y_score <= hi)
            if m.sum():
                ece += (m.sum() / len(y_true)) * abs(y_true[m].mean() - y_score[m].mean())
        return {"Brier": float(brier_score_loss(y_true, y_score)), "ECE": float(ece)}

    @classmethod
    def summary(cls, y_true, y_score, alpha=None, fractions=None, threshold=None):
        """
        Threshold-free metrics always; MCC/F1/accuracy only when a threshold is
        supplied. Pass a threshold chosen on TRAINING folds so the test set is
        never used to select it.

        AUC-PR_lift divides AUC-PR by prevalence: raw AUC-PR moves with class
        balance, so it is not comparable across split regimes whose test sets
        have different active rates (yours are 23.9 / 13.3 / 31.6%).
        """
        from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                                     matthews_corrcoef, accuracy_score, balanced_accuracy_score)
        y_true = np.asarray(y_true).astype(int)
        prev = float(y_true.mean())
        out = {"n": int(len(y_true)), "prevalence": round(prev, 4),
               "ROC-AUC": float(roc_auc_score(y_true, y_score)),
               "AUC-PR":  float(average_precision_score(y_true, y_score)),
               "BEDROC":  cls.bedroc(y_true, y_score, alpha)}
        out["AUC-PR_lift"] = out["AUC-PR"] / prev if prev else np.nan
        out.update(cls.enrichment(y_true, y_score, fractions))
        out.update(cls.calibration(y_true, y_score))
        if threshold is not None:
            p = (np.asarray(y_score) >= threshold).astype(int)
            out.update({"threshold": float(threshold),
                        "MCC": float(matthews_corrcoef(y_true, p)),
                        "F1": float(f1_score(y_true, p)),
                        "Accuracy": float(accuracy_score(y_true, p)),
                        "BalancedAcc": float(balanced_accuracy_score(y_true, p))})
        return out

    @classmethod
    def scorers(cls, alpha=None, fractions=None):
        """Scorer dict for cross_validate. Selection is on BEDROC alone."""
        fr = cls._f(fractions)
        s = {"ROC-AUC": "roc_auc", "AUC-PR": "average_precision",
             "F1": "f1", "MCC": make_scorer(matthews_corrcoef),
             "BEDROC": make_scorer(cls.bedroc, response_method="predict_proba",
                                   alpha=cls._a(alpha))}
        for f in fr:
            s[f"nEF{f*100:g}%"] = make_scorer(cls.nef, response_method="predict_proba",
                                              fraction=f)
        return s

In [ ]:
def tanimoto_kernel(A, B):

    """

    Tanimoto (Jaccard) kernel for binary fingerprints, usable directly as

    SVC(kernel=tanimoto_kernel). RBF on binary vectors is a poor geometric

    match; Tanimoto is the native similarity for fingerprints.

    """

    A = np.asarray(A, dtype=np.float32); B = np.asarray(B, dtype=np.float32)

    inter = A @ B.T

    a = (A * A).sum(1)[:, None]; b = (B * B).sum(1)[None, :]

    den = a + b - inter

    return np.divide(inter, den, out=np.zeros_like(inter), where=den > 0)





class TanimotoNNClassifier(ClassifierMixin, BaseEstimator):   #.. ClassifierMixin FIRST (sklearn >=1.6 tag system)

    """

    The floor. Scores each molecule by its maximum Tanimoto similarity to any

    ACTIVE in the training set. No learned parameters.



    This is the single most important row in your results table. If a tuned

    ensemble cannot beat similarity search, the model is decorative. Published

    audits have found this baseline matching state-of-the-art 3D encoders on

    standard virtual-screening benchmarks, so it is not a straw man.



    Report it separately from the other models. It is a reference line, not a peer.

    """



    def __init__(self, k=1, binarize=True):        #.. new: works on count fingerprints too

        self.k = k

        self.binarize = binarize                   #.. Tanimoto is only exact on binary vectors;

                                                   #   for counts we binarise rather than silently

                                                   #   computing the continuous generalisation



    def _prep(self, X):                            #.. new

        X = np.asarray(X, dtype=np.float32)

        return (X > 0).astype(np.float32) if self.binarize else X



    def fit(self, X, y):

        X = self._prep(X); 

        y = np.asarray(y).astype(int)   #..

        self.classes_ = np.unique(y)

        self.X_act_ = X[y == 1]

        if self.X_act_.shape[0] == 0:

            self.X_act_ = X[:1]         # degenerate fold guard

        return self



    def decision_function(self, X):

        S = tanimoto_kernel(self._prep(X), self.X_act_)    #..

        k = min(self.k, S.shape[1])

        return np.sort(S, axis=1)[:, -k:].mean(axis=1)



    def predict_proba(self, X):

        s = self.decision_function(X)

        return np.column_stack([1 - s, s])



    def predict(self, X):

        return (self.decision_function(X) >= 0.5).astype(int)

In [ ]:
class BioactivityModel:
    """
    QSAR benchmarking, tuning and validation.

    Changes from the thesis version:
      - selection is by mean rank over BEDROC / nEF1% / AUC-PR, not by ROC-AUC
      - every candidate gets the SAME RandomizedSearchCV budget in the screening
        round, so the comparison is not rigged against hyperparameter-sensitive
        algorithms (boosting, SVM, kNN)
      - screening round is nested: tuning happens inside each outer fold
      - fold-level scores are retained for paired significance testing
      - survivors are selected by a 1-SD rule CONFIRMED by paired Wilcoxon
      - grouped CV when the outer split is scaffold- or cluster-based
    """

    # RANKING_METRICS = ["BEDROC", "nEF1%", "AUC-PR"]      #.. metrics used to RANK; the rest are reported only
    RANKING_METRICS = ["BEDROC"]      #.. metrics used to RANK; the rest are reported only

    def __init__(self, cfg=None, random_seed=42, budget=None):
        self.cfg = cfg
        self.seed = random_seed
        self.budget = budget or dict(N_JOBS=-1, SCREEN_ITER=10, TUNE_ITER=100,
                                     N_REPEATS=1, N_SPLITS=5,
                                     Y_SCRAMBLE=10)
        self.results_table = None
        self.fold_scores = None
        self.survivors = []
        self.tuned = {}
        self.best_model_name = None
        self.model = None
        self.opt_threshold = 0.5

    # ---------------- model zoo ----------------
    def build_models(self, fp_name, screen_mode=True):
        """
        Returns {name: Pipeline}. Composition depends on the representation:
          - binary-only models (BernoulliNB, Tanimoto kernel/NN) are omitted for
            count and descriptor representations
          - StandardScaler is prepended for rdkit_desc, INSIDE the pipeline so it
            is fitted per fold rather than on the whole dataset
        Everything is wrapped in a Pipeline named 'clf' so search spaces are uniform.
        """
        s, nj = self.seed, self.budget["N_JOBS"]
        binary = fp_name in ChemUtils.BINARY_FPS
        scale = fp_name in ChemUtils.NEEDS_SCALING
        n_tree = 200 if screen_mode else 500            # cheaper trees during screening

        m = {}
        m["NN_Tanimoto_baseline"] = TanimotoNNClassifier(k=1) if binary else None
        m["BernoulliNB"] = BernoulliNB() if binary else None
        m["SVM_Tanimoto"] = SVC(kernel=tanimoto_kernel, probability=True,
                                class_weight="balanced", random_state=s) if binary else None
        m["LogisticRegression"] = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=s)
        m["KNN"] = KNeighborsClassifier(n_jobs=nj, metric="jaccard" if binary else "minkowski")
        # m["SVM_RBF"] = SVC(probability=True, class_weight="balanced", random_state=s)
        m["RandomForest"] = RandomForestClassifier(n_estimators=n_tree, n_jobs=nj,
                                                   random_state=s, class_weight="balanced")
        m["ExtraTrees"] = ExtraTreesClassifier(n_estimators=n_tree, n_jobs=nj,
                                               random_state=s, class_weight="balanced")
        m["HistGradientBoosting"] = HistGradientBoostingClassifier(random_state=s)
        m["AdaBoost"] = AdaBoostClassifier(random_state=s)
        m["GradientBoosting"] = GradientBoostingClassifier(random_state=s)
        m["MLP"] = MLPClassifier(max_iter=600, random_state=s)
        if HAS_LGBM:
            m["LightGBM"] = LGBMClassifier(random_state=s, n_jobs=nj, verbose=-1, class_weight="balanced")
        if HAS_CAT:
            m["CatBoost"] = CatBoostClassifier(random_seed=s, verbose=0, auto_class_weights="Balanced")

        out = {}
        for k, v in m.items():
            if v is None:
                continue
            steps = [("scale", StandardScaler())] if scale else []
            out[k] = Pipeline(steps + [("clf", v)])
        return out

    def search_space(self, name, n_features):
        """
        SMOKE mode collapses every grid to a single point so RandomizedSearchCV
        fits one configuration (1 candidate x N_SPLITS folds + 1 refit). Use it
        only to verify the pipeline runs end to end. The values are the defaults
        already established as good, so nothing is being "tuned" in this mode.
        """
        if SMOKE:                                              #.. set in the config cell
            return {
                "RandomForest":         {"clf__n_estimators": [400],
                                         "clf__max_features": ["sqrt"],
                                         "clf__min_samples_leaf": [1],
                                         "clf__class_weight": [None]},
                "ExtraTrees":           {"clf__n_estimators": [400],
                                         "clf__max_features": ["sqrt"]},
                "HistGradientBoosting": {"clf__max_iter": [200],
                                         "clf__learning_rate": [0.1]},
                "LightGBM":             {"clf__n_estimators": [300],
                                         "clf__learning_rate": [0.1]},
                "SVM_RBF":              {"clf__C": [1.0]},
                "SVM_Tanimoto":         {"clf__C": [1.0]},
                "LogisticRegression":   {"clf__C": [1.0]},
                "KNN":                  {"clf__n_neighbors": [5]},
                "BernoulliNB":          {"clf__alpha": [1.0]},
                "MLP":                  {"clf__hidden_layer_sizes": [(128,)]},
                "NN_Tanimoto_baseline": {"clf__k": [1]},
            }.get(name, {})

        return {                                               #.. real grids, unchanged below
            "RandomForest": {"clf__n_estimators": randint(200, 1000),
                                         "clf__max_features": ["sqrt", "log2", 0.1, 0.3],
                                         "clf__min_samples_leaf": randint(1, 6),
                                         "clf__class_weight": [None, "balanced", "balanced_subsample"]},
                        "ExtraTrees": {"clf__n_estimators": randint(200, 1000),
                                       "clf__max_features": ["sqrt", "log2", 0.1, 0.3],
                                       "clf__min_samples_leaf": randint(1, 6),
                                       "clf__class_weight": [None, "balanced"]},
                        "HistGradientBoosting": {"clf__max_iter": randint(100, 600),
                                                 "clf__learning_rate": loguniform(1e-2, 3e-1),
                                                 "clf__max_leaf_nodes": randint(15, 63),
                                                 "clf__l2_regularization": loguniform(1e-3, 1e1)},
                        "GradientBoosting": {"clf__n_estimators": randint(100, 500),
                                             "clf__learning_rate": loguniform(1e-2, 3e-1),
                                             "clf__max_depth": randint(2, 7),
                                             "clf__subsample": uniform(0.6, 0.4)},
                        "LightGBM": {"clf__n_estimators": randint(100, 800),
                                     "clf__learning_rate": loguniform(1e-2, 3e-1),
                                     "clf__num_leaves": randint(15, 127),
                                     "clf__min_child_samples": randint(5, 40),
                                     "clf__reg_lambda": loguniform(1e-3, 1e2)},
                        "CatBoost": {"clf__iterations": randint(200, 800),
                                     "clf__learning_rate": loguniform(1e-2, 3e-1),
                                     "clf__depth": randint(4, 9),
                                     "clf__l2_leaf_reg": loguniform(1e0, 1e2)},
                        "AdaBoost": {"clf__n_estimators": randint(50, 400),
                                     "clf__learning_rate": loguniform(1e-2, 2e0)},
                        "SVM_RBF": {"clf__C": loguniform(1e-2, 1e3), "clf__gamma": ["scale", "auto"]},
                        "SVM_Tanimoto": {"clf__C": loguniform(1e-2, 1e3)},
                        "LogisticRegression": {"clf__C": loguniform(1e-3, 1e2),
                                               "clf__penalty": ["l2"], "clf__solver": ["lbfgs", "liblinear"]},
                        "KNN": {"clf__n_neighbors": randint(1, 25), "clf__weights": ["uniform", "distance"]},
                        "BernoulliNB": {"clf__alpha": loguniform(1e-3, 1e1)},
                        "MLP": {"clf__hidden_layer_sizes": [(128,), (256,), (128, 64), (256, 128)],
                                "clf__alpha": loguniform(1e-6, 1e-1),
                                "clf__learning_rate_init": loguniform(1e-4, 1e-2)},
                        "NN_Tanimoto_baseline": {"clf__k": [1, 3, 5]},
        }.get(name, {})


    # ---------------- benchmark ----------------
    def compare_models(self, X, y, groups=None, fp_name="ecfp6", tune_budget=None,
                       n_repeats=None, verbose=True):
        """
        Nested CV. Each outer fold tunes on its own training portion, so the
        reported score is not contaminated by hyperparameter selection.

        NOTE: the INNER cv is ungrouped StratifiedKFold even when the outer cv is
        grouped, because sklearn does not forward `groups` into a nested searcher
        without metadata routing. The outer fold is what protects the reported
        score; the inner fold only affects which hyperparameters get picked.
        State this in the methods section.
        """
        tune_budget = self.budget["SCREEN_ITER"] if tune_budget is None else tune_budget
        n_repeats = self.budget["N_REPEATS"] if n_repeats is None else n_repeats
        models = self.build_models(fp_name, screen_mode=True)
        scoring = ScreeningMetrics.scorers()          #.. alpha + fractions come from cfg
        rows, folds = [], []

        if verbose:
            print("=" * 70)
            print(f"BENCHMARK | fp={fp_name} | {len(models)} models | seed={self.seed} | "
                  f"{n_repeats}x{self.budget['N_SPLITS']}-fold | search n_iter={tune_budget}")
            print("=" * 70)

        for name, est in models.items():
            t0 = time.time()
            per_fold = {k: [] for k in scoring}
            try:
                for r in range(n_repeats):
                    cv = Splitters.make_cv(self.budget["N_SPLITS"], groups, seed=self.seed + r)
                    space = self.search_space(name, X.shape[1])
                    if tune_budget and space and name != "NN_Tanimoto_baseline":
                        inner = Splitters.make_cv(3, None, seed=self.seed + 100 + r)   #.. see docstring note
                        model = RandomizedSearchCV(est, space, n_iter=tune_budget, cv=inner,
                                                   scoring=scoring["BEDROC"], n_jobs=1,
                                                   random_state=self.seed + r, refit=True)
                    else:
                        model = est
                    cvres = cross_validate(model, X, y, cv=cv, scoring=scoring,
                                           n_jobs=self.budget["N_JOBS"], groups=groups,
                                           error_score="raise")
                    for k in scoring:
                        per_fold[k].extend(cvres[f"test_{k}"].tolist())

                row = {"Model": name}
                for k in scoring:
                    a = np.array(per_fold[k], dtype=float)
                    row[k] = float(np.nanmean(a)); row[k + "_sd"] = float(np.nanstd(a))
                row["secs"] = round(time.time() - t0, 1)
                rows.append(row); folds.append({"Model": name, **per_fold})
                if verbose:
                    nk = ScreeningMetrics.primary_nef_key()
                    print(f"  {name:<24} BEDROC {row['BEDROC']:.3f}±{row['BEDROC_sd']:.3f} | "
                          f"{nk} {row[nk]:.3f} | AUC-PR {row['AUC-PR']:.3f} | "
                          f"ROC-AUC {row['ROC-AUC']:.3f} | {row['secs']}s")
            except Exception as e:
                if verbose:
                    print(f"  {name:<24} FAILED: {type(e).__name__}: {e}")

        df = pd.DataFrame(rows)
        ranks = np.zeros(len(df))
        for m in self.RANKING_METRICS:
            ranks += df[m].rank(ascending=False).values
        df["MeanRank"] = ranks / len(self.RANKING_METRICS)
        df = df.sort_values("MeanRank").reset_index(drop=True)
        self.results_table, self.fold_scores = df, {f["Model"]: f for f in folds}
        self.best_model_name = df.iloc[0]["Model"]

        if verbose:
            print("-" * 70)
            print(df[["Model", "MeanRank", "BEDROC", "AUC-PR",
                      "ROC-AUC", "MCC", "F1"]].round(3).to_string(index=False))
            base = df[df.Model == "NN_Tanimoto_baseline"]
            if len(base):
                b = float(base.iloc[0]["BEDROC"])
                print(f"\n  NN-Tanimoto baseline BEDROC = {b:.3f}; "
                      f"{int((df['BEDROC'] > b).sum())}/{len(df)-1} models beat it")
        return df

    def select_survivors(self, primary="BEDROC", max_n=6, alpha=0.05, verbose=True):
        """
        1-SD rule, then a paired Wilcoxon on fold-level scores. Overlapping error
        bars alone have a much higher false-discovery rate than a paired test,
        which is why both gates are applied.
        """
        df = self.results_table
        best = df.iloc[0]["Model"]
        cut = df[primary].max() - df.loc[df[primary].idxmax(), primary + "_sd"]
        cand = df[(df[primary] >= cut) & (df.Model != "NN_Tanimoto_baseline")]["Model"].tolist()
        keep = []
        for name in cand:
            if name == best:
                keep.append(name); continue
            a = np.array(self.fold_scores[best][primary], dtype=float)
            b = np.array(self.fold_scores[name][primary], dtype=float)
            n = min(len(a), len(b))
            try:
                p = wilcoxon(a[:n], b[:n]).pvalue
            except ValueError:
                p = 1.0                       # identical scores -> not distinguishable
            if p > alpha:
                keep.append(name)
            elif verbose:
                print(f"  dropped {name}: significantly worse than {best} (Wilcoxon p={p:.4f})")
        self.survivors = keep[:max_n]
        if verbose:
            print(f"  survivors ({len(self.survivors)}): {self.survivors}")
        return self.survivors

    # ---------------- tuning and fitting ----------------
    def tune(self, X, y, groups=None, fp_name="ecfp6", names=None, n_iter=None, verbose=True):
        """Full-budget random search on the survivors. Grouped CV is used here."""
        n_iter = self.budget["TUNE_ITER"] if n_iter is None else n_iter
        names = names or self.survivors or [self.best_model_name]
        models = self.build_models(fp_name, screen_mode=False)
        cv = Splitters.make_cv(self.budget["N_SPLITS"], groups, seed=self.seed)
        scorer = ScreeningMetrics.scorers()["BEDROC"]
        best_score, best_name = -np.inf, None
        for name in names:
            space = self.search_space(name, X.shape[1])
            if not space:
                self.tuned[name] = (clone(models[name]).fit(X, y), np.nan); continue
            rs = RandomizedSearchCV(models[name], space, n_iter=n_iter, cv=cv, scoring=scorer,
                                    n_jobs=self.budget["N_JOBS"], random_state=self.seed, refit=True)
            rs.fit(X, y, groups=groups) if groups is not None else rs.fit(X, y)
            self.tuned[name] = (rs.best_estimator_, rs.best_score_)
            if verbose:
                print(f"  {name:<24} CV BEDROC {rs.best_score_:.3f} | {rs.best_params_}")
            if rs.best_score_ > best_score:
                best_score, best_name = rs.best_score_, name
        self.best_model_name = best_name or names[0]
        self.model = self.tuned[self.best_model_name][0]
        if verbose:
            print(f"  CHAMPION: {self.best_model_name} (CV BEDROC {best_score:.3f})")
        return self.model

    def predict_proba(self, X):
        """Ensemble-averaged P(active). Use this everywhere downstream."""
        return self.model.predict_proba(X)[:, 1]

    # ---------------- validation ----------------
    def find_mcc_optimized_threshold(self, X, y, groups=None, verbose=True):
        """
        SUPPLEMENTARY ONLY. The screening pipeline ranks and takes the top N; it
        never thresholds. A threshold calibrated on a ~30%-active training prior
        is meaningless applied to a library whose true active rate is ~0.01%.
        """
        cv = Splitters.make_cv(self.budget["N_SPLITS"], groups, seed=self.seed)
        p = cross_val_predict(self.model, X, y, cv=cv, method="predict_proba",
                              n_jobs=self.budget["N_JOBS"], groups=groups)[:, 1]
        ts = np.linspace(0.05, 0.95, 181)
        mccs = [matthews_corrcoef(y, (p >= t).astype(int)) for t in ts]
        self.opt_threshold = float(ts[int(np.argmax(mccs))])
        if verbose:
            print(f"  MCC-optimal threshold {self.opt_threshold:.3f} (CV MCC {max(mccs):.3f}) "
                  f"[SUPPLEMENTARY - screening uses ranking]")
        return self.opt_threshold

    def y_scrambling(self, X, y, groups=None, n_iter=None, verbose=True):
        """
        Seeded RNG so the result is reproducible (the thesis version used a global
        np.random.permutation and was not). Default raised to 30 iterations.

        Note what this does and does not show: it proves the label-feature link is
        real. It says nothing about train/test similarity, so it cannot detect the
        analog leakage that Splitters.leakage_audit is for.
        """
        n_iter = self.budget["Y_SCRAMBLE"] if n_iter is None else n_iter
        rng = np.random.default_rng(self.seed)
        cv = Splitters.make_cv(self.budget["N_SPLITS"], groups, seed=self.seed)
        p = cross_val_predict(self.model, X, y, cv=cv, method="predict_proba",
                              n_jobs=self.budget["N_JOBS"], groups=groups)[:, 1]
        real = roc_auc_score(y, p)
        out = []
        for _ in range(n_iter):
            ys = rng.permutation(y)
            try:
                ps = cross_val_predict(clone(self.model), X, ys, cv=cv, method="predict_proba",
                                       n_jobs=self.budget["N_JOBS"], groups=groups)[:, 1]
                out.append(roc_auc_score(ys, ps))
            except Exception:
                out.append(np.nan)
        out = np.array(out, dtype=float)
        if verbose:
            print(f"  y-scrambling n={n_iter}: random ROC-AUC {np.nanmean(out):.3f} "
                  f"± {np.nanstd(out):.3f} vs real {real:.3f} "
                  f"(delta {real - np.nanmean(out):.3f})")
        return dict(real=real, scrambled=out)

    def seed_sensitivity(self, X, y, groups=None, n=None, verbose=True):
        """
        Varies ONLY the seed and reports the spread. This turns the seed effect you
        observed into a reported result instead of a temptation.

        Read it against your model gaps: if best-worst here is comparable to the
        difference between your top models, that difference is not real and the
        paper should say so.
        """
        n = self.budget["SEED_SENS"] if n is None else n
        sc = ScreeningMetrics.scorers()
        vals = {"BEDROC": [], "ROC-AUC": []}
        for i in range(n):
            m = clone(self.model)
            for _, obj in m.steps:
                for attr in ("random_state", "random_seed"):
                    if hasattr(obj, attr):
                        setattr(obj, attr, self.seed + i)
            cv = Splitters.make_cv(self.budget["N_SPLITS"], groups, seed=self.seed + i)
            r = cross_validate(m, X, y, cv=cv, groups=groups,
                               scoring={"BEDROC": sc["BEDROC"], "ROC-AUC": sc["ROC-AUC"]},
                               n_jobs=self.budget["N_JOBS"])
            vals["BEDROC"].append(r["test_BEDROC"].mean())
            vals["ROC-AUC"].append(r["test_ROC-AUC"].mean())
        if verbose:
            for k, v in vals.items():
                v = np.array(v)
                print(f"  seed sensitivity {k}: mean {v.mean():.4f} sd {v.std():.4f} "
                      f"range [{v.min():.4f}, {v.max():.4f}] (best-worst {v.max()-v.min():.4f})")
        return {k: np.array(v) for k, v in vals.items()}

In [ ]:
class FingerprintScreen:

    """

    Stage 1: pick the representation before spending the full model budget.



    Screens each representation with THREE mechanistically different fast models

    rather than one. Fingerprint choice interacts with algorithm choice, and the

    descriptor choice is often more important than the algorithm choice, so a

    fingerprint ranked by RandomForest alone may not be the best for SVM.



    If the ranking agrees across all three, the choice is robust and you can say

    so in the paper. If it disagrees, carry the top two forward and report the

    disagreement rather than hiding it.

    """



    FAST = ["RandomForest", "LogisticRegression", "NN_Tanimoto_baseline"]



    def __init__(self, cfg=None, seed=42, budget=None):

        self.cfg, self.seed = cfg, seed

        self.budget = budget or dict(N_JOBS=-1, N_SPLITS=5)

        self.table = None



    def run(self, smiles, y, fp_names, bits=2048, groups=None, primary="BEDROC", verbose=True):

        """No tuning here on purpose: defaults, equal footing, cheap."""

        scoring = ScreeningMetrics.scorers()

        rows = []

        for fp in fp_names:

            X, ok = ChemUtils.get_fingerprint(fp, smiles, bits)

            yy = np.asarray(y)[ok]

            gg = None if groups is None else np.asarray(groups)[ok]

            models = BioactivityModel(self.cfg, self.seed, self.budget).build_models(fp, screen_mode=True)

            for name in self.FAST:

                if name not in models:            # baseline/NB skipped for count and descriptor reps

                    continue

                cv = Splitters.make_cv(self.budget.get("N_SPLITS", 5), gg, seed=self.seed)

                t0 = time.time()

                try:

                    r = cross_validate(models[name], X, yy, cv=cv, groups=gg,

                                       scoring=scoring, n_jobs=self.budget["N_JOBS"])

                    rows.append({"fp": fp, "model": name, "n_features": X.shape[1],

                                 **{k: float(np.nanmean(r[f"test_{k}"])) for k in scoring},

                                 "secs": round(time.time() - t0, 1)})

                except Exception as e:

                    print(f"  {fp}/{name} FAILED: {type(e).__name__}: {e}")

            if verbose:

                print(f"  screened {fp} ({X.shape[1]} features)")



        df = pd.DataFrame(rows)

        piv = df.pivot(index="fp", columns="model", values=primary)

        piv["mean_rank"] = piv.rank(ascending=False).mean(axis=1)

        piv = piv.sort_values("mean_rank")

        self.table = df

        if verbose:

            print("\n" + piv.round(3).to_string())

            cons = piv.drop(columns="mean_rank").rank(ascending=False)

            print(f"\n  max rank disagreement across models: "

                  f"{(cons.max(axis=1) - cons.min(axis=1)).max():.0f} positions")

        return piv

### 1.1.1 Model bundle + file-based handoff

In [ ]:
class ModelBundle:
    """
    Self-contained, picklable screening artifact. ONE PER REPRESENTATION.

    Carries everything a screen needs: the fitted pipeline, which representation
    it expects, the bit length, and the training structures required for the
    applicability domain. After a kernel restart you load bundles from disk and
    screen COCONUT without re-running any ML.

    Trained on 100% of the curated data. The split regimes exist to ESTIMATE
    generalisation and each discards 20%; none of them is the model you deploy.
    Reported metrics stay the held-out numbers from the evaluation stage, so this
    refit introduces no optimistic bias into anything published.
    """

    def __init__(self, fp_name, bits, model, train_smiles, train_labels, meta=None):
        self.fp_name      = fp_name
        self.bits         = bits
        self.model        = model
        self.train_smiles = np.asarray(train_smiles)
        self.train_labels = np.asarray(train_labels)
        self.meta         = meta or {}

    def featurize(self, smiles):
        """Bit length travels WITH the model, so no global config is consulted."""
        return ChemUtils.get_fingerprint(self.fp_name, smiles, self.bits)

    def predict_proba(self, X):
        return self.model.predict_proba(X)[:, 1]

    def score_smiles(self, smiles):
        X, ok = self.featurize(smiles)
        return self.predict_proba(X), ok

    def save(self, path):
        joblib.dump(self, path)
        print(f"  saved bundle -> {Path(path).name}  "
              f"({self.meta.get('n_train','?')} compounds, {self.fp_name})")
        return path

    @staticmethod
    def load(path):
        b = joblib.load(path)
        print(f"  loaded {b.fp_name} | n={b.meta.get('n_train','?')} | "
              f"hp from '{b.meta.get('hp_source','?')}'")
        return b


def fit_bundle(fp_name, cfg, budget, champion=None, hp_params=None,
               hp_source="scaffold", verbose=True):
    """
    Fit the production model for one representation.

    The estimator CLASS now comes from the benchmark champion for that
    representation, not a hardcoded RandomForest. HistGradientBoosting beat RF
    on Avalon in both regimes; hardcoding RF silently deployed the wrong model.
    """
    data = pd.read_csv(cfg.CURATED_PATH)
    X, ok = ChemUtils.get_fingerprint(fp_name, data["std_smiles"], cfg.FP_BITS)
    d = data[ok].reset_index(drop=True)
    y = d["active"].values

    if champion is None:                                   #.. read it from the saved benchmark
        bench = pd.read_csv(cfg.RESULTS_DIR / f"benchmark_{fp_name}_{hp_source}.csv")
        bench = bench[bench.Model != "NN_Tanimoto_baseline"]
        champion = bench.sort_values("MeanRank").iloc[0]["Model"]

    est = BioactivityModel(cfg, cfg.SEED, budget).build_models(fp_name, screen_mode=False)[champion]
    if hp_params:
        try:
            est.set_params(**{k: v for k, v in hp_params.items() if k.startswith("clf__")})
        except ValueError as e:
            print(f"  hyperparameters do not apply to {champion}; using defaults ({e})")
    est.fit(X, y)

    b = ModelBundle(fp_name, cfg.FP_BITS, est, d["std_smiles"].values, y,
                    meta={"n_train": int(len(y)), "active_rate": float(y.mean()),
                          "model": champion, "hp_source": hp_source, "seed": cfg.SEED,
                          "domain": cfg.DOMAIN})
    if verbose:
        print(f"  {fp_name}: champion = {champion}")
    return b.save(cfg.bundle_path(fp_name))



def save_json(obj, path):
    """Numpy-safe JSON writer, so every stage can hand off through files."""
    def enc(o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray):     return o.tolist()
        if isinstance(o, (np.bool_,)):    return bool(o)
        raise TypeError(str(type(o)))
    Path(path).write_text(json.dumps(obj, indent=2, default=enc))
    print(f"  wrote {Path(path).name}")
    return path


def load_json(path):
    return json.loads(Path(path).read_text())

In [ ]:
class DomainParser:

    """

    ChEMBL maps targets to UniProt accessions, not to domains, so CHEMBL1163125

    is all of BRD4. The BD1/BD2 distinction lives in assay_description and has to

    be parsed out of free text.



    Two signals are combined:

      1. keywords  ("first bromodomain", "BRD4-2", "site 1", "C-terminal")

      2. residue ranges, classified numerically rather than by literal regex:

           span > 250            -> tandem construct  (1-477, 44-460)

           start 35-80,  end 150-200 -> BD1  (44-168, 49-170, 57-168)

           start 320-375, end 440-490 -> BD2  (333-460, 349-460)



    Numeric classification generalises to construct boundaries you have not seen,

    which literal patterns do not.



    Anything mentioning both domains without the word "tandem" is AMBIGUOUS, not

    BD1. Those need manual review at assay_chembl_id level: ChEMBL creates separate

    assay rows per construct but reuses one description string, so the text alone

    cannot resolve them. Review them or drop them; do not default them to BD1.

    """



    RE_TANDEM = re.compile(r"(tandem|full[-\s]?length|both bromodomains)", re.I)

    # Tyr->Ala knockouts on a 1-477 construct: the DEAD domain names the assay's counterpart   #.. new

    RE_MUT_BD1_DEAD = re.compile(r"(BD1|bromodomain[-\s]?1|first bromodomain)[^.;]{0,40}Y\s*97\s*A|"

                                 r"Y\s*97\s*A[^.;]{0,40}(BD1|mutant)", re.I)      #.. Y97A dead -> BD2 measured

    RE_MUT_BD2_DEAD = re.compile(r"(BD2|bromodomain[-\s]?2|second bromodomain)[^.;]{0,40}Y\s*390\s*A|"

                                 r"Y\s*390\s*A[^.;]{0,40}(BD2|mutant)", re.I)     #.. Y390A dead -> BD1 measured



    RE_BD1_KW = re.compile(r"(first bromodomain|bromodomain[-\s]?1\b|\bBD[-\s]?1\b|BRD4\s*\(\s*1\s*\)|"

                           r"BRD4[-\s]?BD1|BRD4[-\s]?1\b|N[-\s]?terminal bromodomain|site\s*1\b|"

                           r"BRD4\s*\(?\s*D1\s*\)?|BRD4[-\s]?D1\b)", re.I)        #.. "BRD4 (D1)" / "BRD4 D1"

    RE_BD2_KW = re.compile(r"(second bromodomain|bromodomain[-\s]?2\b|\bBD[-\s]?2\b|BRD4\s*\(\s*2\s*\)|"

                           r"BRD4[-\s]?BD2|BRD4[-\s]?2\b|C[-\s]?terminal bromodomain|site\s*2\b|"

                           r"BRD4\s*\(?\s*D2\s*\)?|BRD4[-\s]?D2\b)", re.I)        #..



    # also matches "44 to 168 residues" and "residues N44 to E168", not just "(44-168)"   #.. new

    RE_RANGE = re.compile(r"[A-Z]?\s*(\d{1,3})\s*(?:[-\u2013]|\s+to\s+[A-Z]?)\s*(\d{1,3})")



    BD1_START, BD1_END = (35, 80),   (150, 200)

    BD2_START, BD2_END = (320, 375), (440, 490)



    @classmethod

    def ranges(cls, text):

        out = {"BD1": 0, "BD2": 0, "TANDEM": 0}

        for a, b in cls.RE_RANGE.findall(str(text)):

            a, b = int(a), int(b)

            if b <= a:

                continue

            if (b - a) > 250:

                out["TANDEM"] += 1

            elif cls.BD1_START[0] <= a <= cls.BD1_START[1] and cls.BD1_END[0] <= b <= cls.BD1_END[1]:

                out["BD1"] += 1

            elif cls.BD2_START[0] <= a <= cls.BD2_START[1] and cls.BD2_END[0] <= b <= cls.BD2_END[1]:

                out["BD2"] += 1

        return out



    @classmethod

    def classify(cls, text):

        """Returns BD1 | BD2 | TANDEM | AMBIGUOUS | UNSPECIFIED."""

    @classmethod

    def classify(cls, text):

        t = str(text)

        if not t.strip() or t.lower() == "nan":

            return "UNSPECIFIED"

        if cls.RE_MUT_BD1_DEAD.search(t):        #.. BD1 knocked out -> the measurement is BD2

            return "BD2"

        if cls.RE_MUT_BD2_DEAD.search(t):        #.. BD2 knocked out -> the measurement is BD1

            return "BD1"

        r = cls.ranges(t)

        bd1 = bool(cls.RE_BD1_KW.search(t)) or r["BD1"] > 0

        bd2 = bool(cls.RE_BD2_KW.search(t)) or r["BD2"] > 0

        tandem = bool(cls.RE_TANDEM.search(t)) or r["TANDEM"] > 0

        if bd1 and bd2:

            return "TANDEM" if tandem else "AMBIGUOUS"

        if tandem and not (bd1 or bd2):

            return "TANDEM"

        if bd1:

            return "BD1"

        if bd2:

            return "BD2"

        return "UNSPECIFIED"



    @classmethod

    def annotate(cls, df, col="assay_description", verbose=True):

        """

        Adds a `domain` column and prints counts by ASSAY and by unique COMPOUND.

        The compound count is the one that matters; assay counts mislead badly

        because patent assays test one compound and J Med Chem assays test sixty.

        """

        out = df.copy()

        out["domain"] = out[col].map(cls.classify)

        if verbose:

            a = out.drop_duplicates("assay_chembl_id") if "assay_chembl_id" in out else out

            print("  domain by ASSAY:\n" + a["domain"].value_counts().to_string())

            if "molecule_chembl_id" in out:

                print("  domain by unique COMPOUND:\n" +

                      out.groupby("domain")["molecule_chembl_id"].nunique().to_string())

        return out



    @classmethod

    def review_unresolved(cls, df, col="assay_description", n=40):

        """Run this and READ the output. Manual triage is an hour and is not optional."""

        u = df[df.domain.isin(["UNSPECIFIED", "AMBIGUOUS"])]

        u = u.drop_duplicates("assay_chembl_id") if "assay_chembl_id" in u else u

        return u[col].str.slice(0, 160).value_counts().head(n)





class ChEMBLCurator:

    """

    Replaces the thesis preprocessing cell.



    What changed and why:

      - the old groupby("canonical_smiles").agg(...) collapsed the frame to three

        columns, destroying assay_description and making BD1 selection impossible

      - deduplication now happens on standardised InChIKey, not on the raw SMILES

        string, so salts, charge states and tautomers of one compound collapse

      - replicate aggregation is median, not mean

      - assay_type / data_validity_comment / potential_duplicate filters added

      - the Lipinski Ro5 filter is GONE. It deleted measured actives from the

        training data. Ro5 is a library filter, and natural products break it

        routinely, so it is computed as a column and never used to drop rows

      - PAINS and Brenk are columns, not filters, for the same reason: JQ1 and

        I-BET762 pass both, but tetrahydroxyflavone (the one literature natural

        product BET binder) is flagged by both

    """



    # ordered: first match wins, so the catch-all must stay last

    ASSAY_TECH = {

        "TR-FRET":        r"(TR[-\s]?FRET|time[-\s]?resolved fluorescence)",

        "AlphaScreen":    r"(alpha[-\s]?screen|alpha[-\s]?assay|alphalisa)",

        "Peptide displacement": r"(peptide displacement|displacement of .{0,30}peptide)",

        "ITC":            r"\bITC\b|isothermal titration",

        "SPR":            r"\bSPR\b|surface plasmon",

        "Thermal shift":  r"thermal shift|\bTSA\b|differential scanning fluorimetry",

        "Fluorescence polarisation": r"fluorescence polari",

        "Cell/other":     r".",

    }



    def __init__(self, cfg):

        self.cfg = cfg

        self.report = {}



    @staticmethod

    def _tech(text):

        t = str(text)

        for name, pat in ChEMBLCurator.ASSAY_TECH.items():

            if re.search(pat, t, re.I):

                return name

        return "Cell/other"



    def load_and_filter(self, path=None, verbose=True):

        """Every filter prints its before/after count. Those numbers go in the methods section."""

        path = path or self.cfg.DATA_PATH

        df = pd.read_csv(path, low_memory=False)

        self.report["raw_rows"] = len(df)

        if verbose:

            print(f"  raw rows: {len(df)}")

        if "activity_id" in df:

            df = df.drop_duplicates(subset="activity_id")     # your raw export contains repeated rows

            if verbose:

                print(f"  after activity_id dedup: {len(df)}")



        checks = [

            ("assay_type == B",           "assay_type",            lambda c: c.eq("B")),

            ("standard_relation == '='",  "standard_relation",     lambda c: c.eq("=")),

            ("standard_type IC50/Ki",     "standard_type",         lambda c: c.isin(["IC50", "Ki"])),

            ("data_validity_comment null","data_validity_comment", lambda c: c.isna()),

            ("potential_duplicate == 0",  "potential_duplicate",   lambda c: c.fillna(0).eq(0)),

            ("organism human",            "target_organism",       lambda c: c.eq("Homo sapiens")),

            ("pchembl not null",          "pchembl_value",         lambda c: c.notna()),

            ("smiles not null",           "canonical_smiles",      lambda c: c.notna()),

        ]

        for label, col, fn in checks:

            if col not in df.columns:

                print(f"    SKIP {label}: column '{col}' absent")

                continue

            before = len(df)

            df = df[fn(df[col]).reindex(df.index).fillna(False)]

            if verbose:

                print(f"    {label:<30} {before} -> {len(df)}")



        df["assay_tech"] = df["assay_description"].map(self._tech)

        if verbose:

            print("  assay technology mix:\n" +

                  df.drop_duplicates("assay_chembl_id")["assay_tech"].value_counts().to_string())

        self.report["after_filters"] = len(df)

        return df.reset_index(drop=True)



    def select_domain(self, df, domain=None, verbose=True):

        domain = domain or self.cfg.DOMAIN

        df = DomainParser.annotate(df, verbose=verbose)

        if domain == "ALL":

            return df

        out = df[df.domain == domain].reset_index(drop=True)

        if verbose:

            n = out["molecule_chembl_id"].nunique() if "molecule_chembl_id" in out else "?"

            print(f"  domain={domain}: {len(out)} rows, {n} unique compounds")

        self.report[f"rows_{domain}"] = len(out)

        return out



    def label(self, df, verbose=True):

        """

        Grey-zone exclusion. Justify it by measurement noise, not convenience:

        paired IC50 values for the same compound and target disagree by more than

        0.3 log units in ~65% of cases under minimal curation.



        Run once with DROP_GREY_ZONE=True and once False on the final model only.

        Dropping the grey zone removes your hardest examples and inflates every

        metric for free, so report both.

        """

        cfg = self.cfg

        lo, hi = cfg.INACTIVE_PCHEMBL, cfg.ACTIVE_PCHEMBL

        df = df.copy()

        df["activity_class"] = np.where(df.pchembl_value >= hi, "ACTIVE",

                                np.where(df.pchembl_value < lo, "INACTIVE", "GREY"))

        if verbose:

            print(f"  labels (active>={hi}, inactive<{lo}): {df.activity_class.value_counts().to_dict()}")

        if cfg.DROP_GREY_ZONE:

            df = df[df.activity_class != "GREY"].reset_index(drop=True)

            if verbose:

                print(f"  grey zone dropped -> {len(df)} compounds")

        else:

            df["activity_class"] = np.where(df.pchembl_value >= (lo + hi) / 2, "ACTIVE", "INACTIVE")

            if verbose:

                print(f"  grey zone KEPT, single cut at {(lo+hi)/2} -> {len(df)} compounds")

        df["active"] = (df.activity_class == "ACTIVE").astype(int)

        self.report["n_active"] = int(df.active.sum())

        self.report["n_inactive"] = int((1 - df.active).sum())

        return df



    def run(self, path=None, domain=None, verbose=True):

        print("=" * 70); print(f"CURATION | {self.cfg.PROTEIN_NAME} | domain={domain or self.cfg.DOMAIN}")

        print("=" * 70)

        df = self.load_and_filter(path, verbose)

        df = self.select_domain(df, domain, verbose)

        if len(df) == 0:

            raise ValueError("no rows after domain filter - check DomainParser.review_unresolved()")

        df = ChemStandardizer.standardize_frame(df, "canonical_smiles", verbose=verbose)

        self.report["replicates"] = ChemStandardizer.replicate_spread(

            df, "inchikey", "pchembl_value", self.cfg.INACTIVE_PCHEMBL, self.cfg.ACTIVE_PCHEMBL, verbose)

        df = ChemStandardizer.deduplicate(df, "pchembl_value", "median", verbose)

        df = self.label(df, verbose)

        df = ChemUtils.annotate_alerts(df, "std_smiles")

        desc = df["std_smiles"].apply(ChemUtils.calculate_descriptors)

        df = pd.concat([df.reset_index(drop=True), desc.reset_index(drop=True)], axis=1)



        if verbose:

            print(f"  PAINS flagged: {(df.pains_status == 'FLAG').sum()} | "

                  f"Brenk flagged: {(df.brenk_status == 'FLAG').sum()} | "

                  f"RO5 viol > 1: {(df.RO5_violations > 1).sum()}   (all REPORTED, none dropped)")

            print(f"  FINAL: {len(df)} compounds | {df.active.sum()} active "

                  f"({df.active.mean():.1%}) | {len(df) - df.active.sum()} inactive")



        cols = [c for c in ["molecule_chembl_id", "std_smiles", "canonical_smiles", "inchikey",

                            "inchikey_skeleton", "pchembl_value", "activity_class", "active",

                            "domain", "assay_tech", "pains_status", "pains_alerts",

                            "brenk_status", "brenk_alerts", "MW", "LogP", "HBD", "HBA",

                            "RotBonds", "TPSA", "HeavyAtoms", "RingCount", "FracCSP3",

                            "QED", "RO5_violations"] if c in df.columns]

        out = df[cols]

        out.to_csv(self.cfg.CURATED_PATH, index=False)

        print(f"  saved -> {self.cfg.CURATED_PATH}")

        return out

### from evalution.py

In [ ]:
PUB_STYLE = {
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.linewidth": 0.8, "axes.grid": False, "figure.dpi": 120,
    "savefig.dpi": 300, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.2,
}
GREYSCALE = False
PALETTE = ["#000000", "#555555", "#999999", "#cccccc"] if GREYSCALE else \
          ["#1f3b73", "#c1121f", "#2a9d8f", "#e09f3e", "#6a4c93", "#8d99ae"]


class Figures:
    """
    Eight figures, PNG only at 300 dpi (PDF doubled the file count for no gain;
    add `fig.savefig(..., format="pdf")` at submission time if a journal asks).

    Every figure is driven by a SAVED ARTIFACT (csv / json / npz), never by a
    variable left in memory, so plots can be regenerated or restyled without
    retraining anything.

    Dropped from the earlier version: fig_scaffolds, fig_confusion,
    fig_fp_overlap, fig_roc_pr (its PR panel is folded into fig_enrichment),
    fig_screening_scores. All were either supplementary or one-line tables.
    """

    def __init__(self, cfg, style=True):
        self.cfg = cfg
        self.dir = cfg.FIGURE_DIR
        if style:
            plt.rcParams.update(PUB_STYLE)

    def _save(self, fig, name):
        p = f"{self.dir}/{name}.png"
        fig.savefig(p)
        plt.close(fig)
        print(f"    saved {name}.png")
        return p

    # ---------------- EDA ----------------
    def fig_activity_distribution(self, df_all, name="eda_activity_distribution"):
        """pChEMBL histogram with the cutoff marked. Pass the PRE-exclusion frame."""
        lo, hi = self.cfg.INACTIVE_PCHEMBL, self.cfg.ACTIVE_PCHEMBL
        fig, ax = plt.subplots(figsize=(4.2, 2.8))
        ax.hist(df_all.pchembl_value, bins=50, color=PALETTE[0], alpha=.85,
                edgecolor="white", lw=.3)
        if hi > lo:
            ax.axvspan(lo, hi, color=PALETTE[1], alpha=.15)
            n_grey = int(((df_all.pchembl_value >= lo) & (df_all.pchembl_value < hi)).sum())
            ax.text((lo + hi) / 2, ax.get_ylim()[1] * .92, f"excluded\nn={n_grey}",
                    ha="center", va="top", fontsize=7, color=PALETTE[1])
        ax.axvline(hi, color=PALETTE[1], ls="--")
        ax.text(hi, ax.get_ylim()[1] * .98, f" active >= {hi:g}", fontsize=7,
                va="top", color=PALETTE[1])
        ax.set_xlabel("pChEMBL (-log10 IC50)"); ax.set_ylabel("compounds")
        return self._save(fig, name)

    def fig_descriptor_boxplots(self, df, cols=("MW", "LogP", "HBD", "HBA", "TPSA", "RotBonds"),
                                name="eda_descriptors"):
        """Returns the Mann-Whitney table. This is the correct use of Ro5 descriptors:
        compute, plot, test. Never filter the training set on them."""
        fig, axes = plt.subplots(2, 3, figsize=(7.2, 4.2))
        stats = []
        for ax, c in zip(axes.ravel(), cols):
            a = df.loc[df.active == 1, c].dropna(); i = df.loc[df.active == 0, c].dropna()
            ax.boxplot([i, a], tick_labels=["inactive", "active"], widths=.55, patch_artist=True,
                       boxprops=dict(facecolor=PALETTE[2], alpha=.6, lw=.8),
                       medianprops=dict(color="black", lw=1.1))
            try:
                p = mannwhitneyu(a, i, alternative="two-sided").pvalue
            except ValueError:
                p = np.nan
            stats.append({"descriptor": c, "median_active": a.median(),
                          "median_inactive": i.median(), "mannwhitney_p": p})
            ax.set_title(f"{c}  p={p:.2e}" if np.isfinite(p) else c)
        fig.tight_layout(); self._save(fig, name)
        return pd.DataFrame(stats)

    def fig_chemical_space(self, train_smiles, test_smiles, library_smiles=None,
                           n_lib=3000, seed=42, name="eda_chemical_space"):
        """PCA of ECFP4 for train / test / library. Carries the AD argument in one image."""
        rng = np.random.default_rng(seed)
        sets = [("training", np.asarray(train_smiles)), ("held-out test", np.asarray(test_smiles))]
        if library_smiles is not None:
            lib = np.asarray(library_smiles)
            sets.append(("library sample",
                         lib[rng.choice(len(lib), min(n_lib, len(lib)), replace=False)]))
        Xs, labs = [], []
        for lab, s in sets:
            X, ok = ChemUtils.get_fingerprint("ecfp4", s, 2048)
            Xs.append(X); labs += [lab] * X.shape[0]
        X = np.vstack(Xs); labs = np.array(labs)
        P = PCA(n_components=2, random_state=seed).fit(X); Z = P.transform(X)
        fig, ax = plt.subplots(figsize=(4.2, 3.4))
        for k, (lab, _) in enumerate(sets):
            m = labs == lab
            ax.scatter(Z[m, 0], Z[m, 1], s=6, alpha=.45 if lab == "library sample" else .8,
                       label=f"{lab} (n={m.sum()})", color=PALETTE[k], linewidths=0)
        ax.set_xlabel(f"PC1 ({P.explained_variance_ratio_[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({P.explained_variance_ratio_[1]*100:.1f}%)")
        ax.legend(frameon=False, loc="best")
        return self._save(fig, name)


    def fig_confusion(self, y_true, y_score, threshold, name="res_confusion"):
            """
            Supplementary. The screening pipeline never thresholds, so state the
            threshold in the caption and note it was chosen on the TRAINING folds
            by maximising MCC, not on the test set.
            """
            from sklearn.metrics import confusion_matrix, matthews_corrcoef, f1_score
            y_true = np.asarray(y_true).astype(int)
            pred = (np.asarray(y_score) >= threshold).astype(int)
            cm = confusion_matrix(y_true, pred)
            tn, fp, fn, tp = cm.ravel()
            fig, ax = plt.subplots(figsize=(3.2, 2.8))
            sns.heatmap(cm, annot=True, fmt="d", cbar=False, cmap="Greys", ax=ax,
                        xticklabels=["pred. inactive", "pred. active"],
                        yticklabels=["inactive", "active"], annot_kws={"fontsize": 10})
            ax.set_title(f"t={threshold:.3f} | MCC {matthews_corrcoef(y_true,pred):.3f} | "
                        f"F1 {f1_score(y_true,pred):.3f}\n"
                        f"sens {tp/(tp+fn):.2f}  spec {tn/(tn+fp):.2f}  prec {tp/(tp+fp):.2f}",
                        fontsize=8)
            return self._save(fig, name)



    def fig_leakage(self, audits, name="eda_leakage_maxtc"):
        """
        max-Tc(test -> train) per split regime. The separation between curves IS
        the leakage evidence. audits = {regime: {"max_tc": array, ...}} loaded
        from the saved npz, so this never requires re-splitting.
        """
        fig, ax = plt.subplots(figsize=(4.2, 2.8))
        for k, (regime, rep) in enumerate(audits.items()):
            tc = np.asarray(rep["max_tc"])
            ax.hist(tc, bins=40, histtype="step", lw=1.4, density=True,
                    color=PALETTE[k], label=f"{regime} (median {np.median(tc):.2f})")
        ax.axvline(0.6, color="black", ls=":", lw=.9)
        ax.text(0.6, ax.get_ylim()[1] * .95, " analogue\n flag", fontsize=7, va="top")
        ax.set_xlabel("max Tanimoto, test compound to training set")
        ax.set_ylabel("density"); ax.legend(frameon=False)
        return self._save(fig, name)

    # ---------------- results ----------------
    def fig_model_benchmark(self, fold_scores, metric=None,
                            baseline="NN_Tanimoto_baseline", name="res_model_benchmark"):
        """Per-fold box plot with the similarity baseline as a reference line."""
        metric = metric or self.cfg.RANKING_METRIC
        rows = [(m, v) for m, d in fold_scores.items() for v in d[metric]]
        df = pd.DataFrame(rows, columns=["Model", metric])
        order = df.groupby("Model")[metric].mean().sort_values(ascending=False).index.tolist()
        fig, ax = plt.subplots(figsize=(4.6, max(2.4, .30 * len(order))))
        sns.boxplot(data=df, y="Model", x=metric, order=order, ax=ax,
                    color=PALETTE[2], fliersize=2, linewidth=.8)
        if baseline in fold_scores:
            b = float(np.mean(fold_scores[baseline][metric]))
            ax.axvline(b, color=PALETTE[1], ls="--", lw=1.1)
            ax.text(b, -.6, " baseline", color=PALETTE[1], fontsize=7, va="bottom")
        ax.set_ylabel(""); ax.set_xlabel(f"{metric} (per fold)")
        return self._save(fig, name)

    def fig_enrichment(self, y_true, y_score, name="res_enrichment"):
        """
        Accumulation curve + precision-recall panel. Replaces the separate
        fig_roc_pr. Title reads its nEF key from ScreeningMetrics rather than a
        hardcoded 'nEF1%', which is what broke when the fractions changed.
        """
        y = np.asarray(y_true)[np.argsort(-np.asarray(y_score))]
        frac = np.arange(1, len(y) + 1) / len(y)
        m = ScreeningMetrics.summary(y_true, y_score)
        nef_key = ScreeningMetrics.primary_nef_key()

        fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))
        axes[0].plot(frac, np.cumsum(y) / y.sum(), color=PALETTE[0], label="model")
        axes[0].plot([0, 1], [0, 1], ls=":", color="black", lw=.8, label="random")
        axes[0].plot(frac, np.minimum(frac * len(y) / y.sum(), 1), ls="--",
                     color=PALETTE[2], lw=.9, label="ideal")
        for f in ScreeningMetrics.FRACTIONS:
            axes[0].axvline(f, color=PALETTE[1], lw=.6, alpha=.5)
        axes[0].set_title(f"BEDROC({ScreeningMetrics.ALPHA:g})={m['BEDROC']:.3f}  "
                          f"{nef_key}={m[nef_key]:.3f}")
        axes[0].set_xlabel("fraction of ranked database")
        axes[0].set_ylabel("fraction of actives found")
        axes[0].legend(frameon=False)

        pr, rc, _ = precision_recall_curve(y_true, y_score)
        axes[1].plot(rc, pr, color=PALETTE[0])
        axes[1].axhline(np.mean(y_true), ls=":", color="black", lw=.8)
        axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
        axes[1].set_title(f"AUC-PR={m['AUC-PR']:.3f} (lift {m['AUC-PR_lift']:.2f}x)")
        fig.tight_layout()
        return self._save(fig, name)

    def fig_validation(self, scramble, seed_sens=None, gap=None, name="res_validation"):
        """
        Two panels: y-scrambling and seed sensitivity. Merged because they answer
        the same question (is the signal real and is the ranking stable) and
        neither needs a full page.
        """
        n = 2 if seed_sens else 1
        fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 2.8))
        axes = np.atleast_1d(axes)

        s = np.asarray(scramble["scrambled"], dtype=float); r = float(scramble["real"])
        axes[0].hist(s, bins=12, color=PALETTE[2], edgecolor="white", lw=.4,
                     label=f"scrambled (n={len(s)}, {np.nanmean(s):.3f}+/-{np.nanstd(s):.3f})")
        axes[0].axvline(r, color=PALETTE[1], lw=1.6, label=f"real ({r:.3f})")
        axes[0].axvline(0.5, color="black", ls=":", lw=.8)
        axes[0].set_xlabel("ROC-AUC"); axes[0].set_ylabel("iterations")
        axes[0].legend(frameon=False, fontsize=7)
        axes[0].set_title("Y-scrambling")

        if seed_sens:
            v = np.asarray(seed_sens[self.cfg.RANKING_METRIC], dtype=float)
            axes[1].plot(range(len(v)), v, "o-", color=PALETTE[0], ms=3)
            axes[1].axhline(v.mean(), color=PALETTE[1], ls="--", lw=.9)
            if gap:
                axes[1].axhspan(v.mean() - gap / 2, v.mean() + gap / 2,
                                color=PALETTE[3], alpha=.35)
            axes[1].set_xlabel("seed index"); axes[1].set_ylabel(self.cfg.RANKING_METRIC)
            axes[1].set_title(f"seed sd {v.std():.4f}, range {v.max()-v.min():.4f}")
        fig.tight_layout()
        return self._save(fig, name)

    # ---------------- screening ----------------
    def fig_applicability_domain(self, scored, selection, name="scr_applicability_domain"):
        """
        Left: Tc-to-training-actives, library vs selection. A right-shifted
        selection means your hits are analogues of known chemotypes, not novel
        natural products. Report it either way.
        Right: AD band composition of the shortlist.
        """
        fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.9))
        axes[0].hist(scored.max_tc_active.dropna(), bins=40, density=True,
                     color=PALETTE[2], alpha=.75, label=f"library (n={len(scored)})")
        axes[0].hist(selection.max_tc_active.dropna(), bins=25, density=True, histtype="step",
                     lw=1.5, color=PALETTE[1], label=f"selection (n={len(selection)})")
        axes[0].axvline(0.35, color="black", ls=":", lw=.8)
        axes[0].set_xlabel("max Tc to training actives"); axes[0].set_ylabel("density")
        axes[0].legend(frameon=False, fontsize=7)
        c = selection.ad_band.value_counts()
        axes[1].bar(range(len(c)), c.values, color=PALETTE[:len(c)])
        axes[1].set_xticks(range(len(c)))
        axes[1].set_xticklabels(c.index, fontsize=7, rotation=15)
        axes[1].set_ylabel("compounds in selection")
        fig.tight_layout()
        return self._save(fig, name)

    def fig_docking_control(self, selected_scores, control_scores,
                            name="dock_vs_random_control"):
        """
        THE control figure. Run after docking. If these distributions overlap,
        the ML prefilter added nothing on out-of-domain chemistry, which is a
        real and reportable result.
        """
        fig, ax = plt.subplots(figsize=(4.0, 3.0))
        ax.hist(control_scores, bins=25, density=True, color=PALETTE[2], alpha=.7,
                label=f"random library (n={len(control_scores)}, "
                      f"med {np.median(control_scores):.2f})")
        ax.hist(selected_scores, bins=25, density=True, histtype="step", lw=1.6,
                color=PALETTE[1],
                label=f"ML-selected (n={len(selected_scores)}, "
                      f"med {np.median(selected_scores):.2f})")
        try:
            p = mannwhitneyu(selected_scores, control_scores, alternative="less").pvalue
            ax.set_title(f"Mann-Whitney p = {p:.2e}")
        except ValueError:
            pass
        ax.set_xlabel("Vina score (kcal/mol)"); ax.set_ylabel("density")
        ax.legend(frameon=False, fontsize=7)
        return self._save(fig, name)

## 1.2 Data, preprocess, filter





> **TODO:** which fixes and why





smiles is array of smiles like "CCNC(=O)C[C@@H]1N=C(c2ccc(Cl)cc2)c2cc(OC)ccc2-n2c(C)nnc21" 

active is: 1 or 0 active inactive respectively

In [ ]:
# Single source of truth for every metric default. Without this call the class
# falls back to its own defaults and cfg.EF_FRACTIONS_CV is silently ignored,
# which is exactly the bug that made the figures crash.
ScreeningMetrics.configure(cfg)
print(f"metrics: alpha={ScreeningMetrics.ALPHA}, CV fractions={ScreeningMetrics.FRACTIONS}, "
      f"screen fractions={ScreeningMetrics.FRACTIONS_SCREEN}")

# ============================================================
# STAGE 1  CURATION          reads: raw csv      writes: curated csv
# ============================================================
FORCE_RECURATE = True

if FORCE_RECURATE or not cfg.CURATED_PATH.exists():
    data = ChEMBLCurator(cfg).run()
else:
    data = pd.read_csv(cfg.CURATED_PATH)
    print(f"loaded curated file: {len(data)} compounds, "
          f"{data.active.sum()} active ({data.active.mean():.1%})")

# Triage the unresolved assays once per target, then comment this back out.
# curator = ChEMBLCurator(cfg)
# raw_ann = DomainParser.annotate(curator.load_and_filter(verbose=False), verbose=False)
# print(DomainParser.review_unresolved(raw_ann, n=40).to_string())

In [ ]:
# ============================================================
# STAGE 2  FINGERPRINT SCREEN    once per target; skip on re-runs
# reads: curated csv            writes: fingerprint_screen.csv
# ============================================================
RUN_FP_SCREEN = False          #.. set True the first time you work on a new target

if RUN_FP_SCREEN:
    FP_CANDIDATES = ["ecfp4_maccs", "maccs", "ecfp4", "ecfp6", "avalon", "rdk7", "rdkit_desc"]
    fp_table = FingerprintScreen(
        cfg, seed=cfg.SEED,
        budget=dict(N_JOBS=BUDGET["N_JOBS"], N_SPLITS=BUDGET["N_SPLITS"])
    ).run(data["std_smiles"].values, data["active"].values, FP_CANDIDATES, bits=cfg.FP_BITS)
    fp_table.to_csv(cfg.RESULTS_DIR / "fingerprint_screen.csv")
else:
    print("fingerprint screen skipped; PRIMARY_FP / SECONDARY_FP set in the config cell")

# NOTE ON SELECTION: the screen showed all representations within noise of one
# another (spread 0.886-0.926, three models disagreeing by up to 4 rank
# positions). TOP_FPS is therefore chosen by FAMILY DISTINCTNESS, because
# consensus only improves enrichment when the methods are genuinely different,
# and nested variants (ecfp4 / ecfp6 / *_count) are not.
print(f"carrying forward: {TOP_FPS}")

In [ ]:
# (moved) representation comparison lives in notes/fp_comparison.py -
# it is a one-off analysis, not part of the pipeline.

In [ ]:
# TOP_FPS is set in the config cell (cell 3). No duplicate definition here.

## 1.3 Exploratory Data Analysis



> **TODO**: Proper complete EDA prepare

<!-- -->

In [ ]:
# (intentionally empty)

## 1.4 Model build, validation, analysis, y-scrambling



> **TODO:** Threshold tuning apply Youden's J on main code

<!-- -->

In [ ]:
# ============================================================
# STAGE 3  EVALUATION        reads: curated csv
# writes: benchmark_*.csv, folds_*.json, leakage_*.npz, test_metrics.json
#
# SCOPE: primary representation on all three regimes (this is where the
# leakage estimate comes from); secondary on random only, just to show it is
# comparable. 4 benchmarks, not 6.
# ============================================================
EVAL_PLAN = {PRIMARY_FP: ["random", "scaffold", "butina"],
             SECONDARY_FP: ["random", "scaffold"]}

test_metrics, tuned_params = {}, {}

for fp_name, regimes_wanted in EVAL_PLAN.items():
    X, ok = ChemUtils.get_fingerprint(fp_name, data["std_smiles"], cfg.FP_BITS)
    d  = data[ok].reset_index(drop=True)
    yy = d["active"].values
    ss = d["std_smiles"].values

    regimes = {}
    if "random" in regimes_wanted:
        regimes["random"] = Splitters._group_split(          #.. group by skeleton, not free shuffle
            d.inchikey_skeleton.values, yy, 0.2, cfg.SEED,
            name="random(stereo-grouped)") + (None,)         #.. returns (tr, te); add groups=None

    if "scaffold" in regimes_wanted: regimes["scaffold"] = Splitters.scaffold_split(ss, yy, 0.2, cfg.SEED)
    if "butina"   in regimes_wanted: regimes["butina"]   = Splitters.butina_split(ss, yy, 0.2, cfg.SEED, cutoff=0.65)

    for regime, (tr, te, groups) in regimes.items():  # tr and te are the index of split which 
        key = f"{fp_name}|{regime}"
        print("\n" + "#" * 70); print(f"# {key}"); print("#" * 70)

        rep = Splitters.leakage_audit(
            ss[tr], ss[te], yy[tr], yy[te],
            inchikey_train=d.inchikey.values[tr],  inchikey_test=d.inchikey.values[te],
            skeleton_train=d.inchikey_skeleton.values[tr],
            skeleton_test=d.inchikey_skeleton.values[te])
        np.savez_compressed(cfg.RESULTS_DIR / f"leakage_{fp_name}_{regime}.npz",
                            max_tc=rep["max_tc"],
                            median_max_tc=rep["median_max_tc"],
                            frac_above_flag=rep["frac_above_flag"],
                            exact_overlap=rep.get("exact_overlap", -1),
                            skeleton_overlap=rep.get("skeleton_overlap", -1),
                            analog_active_pairs=rep.get("analog_active_pairs", -1))

        g_tr = None if groups is None else groups[tr]
        bm = BioactivityModel(cfg, cfg.SEED, BUDGET)
        bm.compare_models(X[tr], yy[tr], groups=g_tr, fp_name=fp_name)
        bm.select_survivors(max_n=6)

        # Tune ONLY the production combination. The other regimes exist to
        # measure leakage, and tuning them costs hours for ~0.006 BEDROC.
        if fp_name == PRIMARY_FP and regime == "scaffold":
            bm.tune(X[tr], yy[tr], groups=g_tr, fp_name=fp_name)
            tuned_params[fp_name] = {k: v for k, v in
                                     bm.model.get_params().items() if k.startswith("clf__")}
        else:
            bm.model = bm.build_models(fp_name, screen_mode=False)[bm.best_model_name]
            bm.model.fit(X[tr], yy[tr])

        # threshold chosen on TRAINING folds, so the test set never selects it
        bm.find_mcc_optimized_threshold(X[tr], yy[tr], groups=g_tr)
        m = ScreeningMetrics.summary(yy[te], bm.predict_proba(X[te]),
                                     threshold=bm.opt_threshold)
        # persist the HELD-OUT predictions so figures never need to re-predict.
        # The production bundle is trained on 100% of the data, so using it for
        # any performance plot is in-sample scoring.
        np.savez_compressed(cfg.RESULTS_DIR / f"preds_{fp_name}_{regime}.npz",
                            y_true=yy[te].astype(np.int8),
                            y_score=bm.predict_proba(X[te]).astype(np.float32),
                            threshold=float(bm.opt_threshold),
                            prevalence=float(yy[te].mean()))

        print("  HELD-OUT TEST: " + " | ".join(
            f"{k} {v:.3f}" if isinstance(v, float) else f"{k} {v}" for k, v in m.items()))

        test_metrics[key] = m
        bm.results_table.to_csv(cfg.RESULTS_DIR / f"benchmark_{fp_name}_{regime}.csv", index=False)
        save_json({mdl: {k: list(map(float, v)) for k, v in sc.items() if k != "Model"}
                   for mdl, sc in bm.fold_scores.items()},
                  cfg.RESULTS_DIR / f"folds_{fp_name}_{regime}.json")
        if fp_name == PRIMARY_FP and regime == "scaffold":
            PROD_BM = bm                      #.. used by STAGE 4 in this session only

save_json(test_metrics, cfg.result_path("test_metrics"))
save_json(tuned_params, cfg.result_path("tuned_params"))

# ---- headline: the random-minus-butina gap is the leakage estimate ----
r = test_metrics.get(f"{PRIMARY_FP}|random", {})
b = test_metrics.get(f"{PRIMARY_FP}|butina", {})
if r and b:
    print(f"\n  LEAKAGE ESTIMATE ({PRIMARY_FP}): BEDROC random {r['BEDROC']:.3f} "
          f"vs butina {b['BEDROC']:.3f} -> {r['BEDROC'] - b['BEDROC']:+.3f}")
    print(f"  (test prevalence differs: {r['prevalence']:.3f} vs {b['prevalence']:.3f}; "
          f"compare AUC-PR_lift {r['AUC-PR_lift']:.2f} vs {b['AUC-PR_lift']:.2f} "
          f"and ROC-AUC {r['ROC-AUC']:.3f} vs {b['ROC-AUC']:.3f}, which are prevalence-robust)")

In [ ]:
# ============================================================
# STAGE 4  VALIDATION on the production combination
# reads: curated csv + PROD_BM     writes: validation.json
# ============================================================
X, ok = ChemUtils.get_fingerprint(PRIMARY_FP, data["std_smiles"], cfg.FP_BITS)
d  = data[ok].reset_index(drop=True)
yy = d["active"].values
tr, te, groups = Splitters.scaffold_split(d["std_smiles"].values, yy, 0.2, cfg.SEED)
g_tr = None if groups is None else groups[tr]

scr = PROD_BM.y_scrambling(X[tr], yy[tr], groups=g_tr)
ssd = PROD_BM.seed_sensitivity(X[tr], yy[tr], groups=g_tr)

top2 = PROD_BM.results_table[cfg.RANKING_METRIC].nlargest(2).values
gap  = float(top2[0] - top2[1]) if len(top2) > 1 else 0.0

save_json({"scrambling": {"real": float(scr["real"]),
                          "scrambled": [float(v) for v in scr["scrambled"]]},
           "seed_sensitivity": {k: [float(x) for x in v] for k, v in ssd.items()},
           "top2_gap": gap,
           "opt_threshold": float(PROD_BM.opt_threshold)},
          cfg.result_path("validation"))

In [ ]:
# ============================================================
# STAGE 5  FIT PRODUCTION BUNDLES    reads: curated csv + tuned_params.json
# writes: one .pkl per representation
#
# Trained on 100% of the curated data. The split regimes estimate
# generalisation and each discards 20%; neither is the model you deploy.
# Reported metrics stay the held-out ones from STAGE 3.
# ============================================================
hp = load_json(cfg.result_path("tuned_params")) if cfg.result_path("tuned_params").exists() else {}

for fp_name in TOP_FPS:
    fit_bundle(fp_name, cfg, BUDGET,
               hp_params=hp.get(fp_name), hp_source="scaffold")

print("\nbundles on disk:")
for p in sorted(cfg.BUNDLE_DIR.glob("*.pkl")):
    print("  ", p.name)

In [ ]:
# ============================================================
# STAGE 7  FIGURES
# Reads saved artifacts only. Nothing here re-trains, and nothing here
# touches the production bundle: that model is fitted on 100% of the curated
# data, so scoring the test split with it is in-sample and produced a fake
# MCC of 0.97. All performance figures come from preds_*.npz, which STAGE 3
# wrote from genuinely held-out predictions.
# ============================================================
F = Figures(cfg)

FIG_REGIME = "scaffold"      #.. which split the results figures depict; put this in every caption

# ---------------- EDA ----------------
cur = ChEMBLCurator(cfg)
raw = cur.select_domain(cur.load_and_filter(verbose=False), verbose=False)
F.fig_activity_distribution(raw)
F.fig_descriptor_boxplots(data).to_csv(
    cfg.RESULTS_DIR / "descriptor_mannwhitney.csv", index=False)

# chemical space: train / test / library. Split reproduced only to colour the
# points; no model is involved.
X, ok = ChemUtils.get_fingerprint(PRIMARY_FP, data["std_smiles"], cfg.FP_BITS)
dp = data[ok].reset_index(drop=True)
tr, te, _ = Splitters.scaffold_split(dp["std_smiles"].values, dp.active.values, 0.2, cfg.SEED)

lib_path = cfg.INPUT_CSV if Path(cfg.INPUT_CSV).exists() else cfg.COCONUT_CSV
lib_sample = pd.read_csv(lib_path, nrows=20000)["canonical_smiles"].dropna().values
F.fig_chemical_space(dp["std_smiles"].values[tr], dp["std_smiles"].values[te], lib_sample)

audits = {}
for regime in ("random", "scaffold", "butina"):
    p = cfg.RESULTS_DIR / f"leakage_{PRIMARY_FP}_{regime}.npz"
    if p.exists():
        audits[regime] = {"max_tc": np.load(p)["max_tc"]}
if audits:
    F.fig_leakage(audits)

# ---------------- results: HELD-OUT predictions only ----------------
pred_path = cfg.RESULTS_DIR / f"preds_{PRIMARY_FP}_{FIG_REGIME}.npz"
if not pred_path.exists():
    raise FileNotFoundError(
        f"{pred_path.name} missing. STAGE 3 must save held-out predictions; "
        f"do NOT substitute the production bundle, it is trained on all data.")

pz    = np.load(pred_path)
y_te  = pz["y_true"]
s_te  = pz["y_score"]
thr   = float(pz["threshold"])
print(f"  results figures from {PRIMARY_FP}|{FIG_REGIME} held-out set: "
      f"n={len(y_te)}, prevalence={y_te.mean():.3f}, threshold={thr:.3f}")

folds = load_json(cfg.RESULTS_DIR / f"folds_{PRIMARY_FP}_{FIG_REGIME}.json")
F.fig_model_benchmark(folds)
F.fig_enrichment(y_te, s_te)
F.fig_confusion(y_te, s_te, thr)

val = load_json(cfg.result_path("validation"))
F.fig_validation(val["scrambling"], val.get("seed_sensitivity"), val.get("top2_gap"))

# consistency check: these must match the STAGE 3 printout for the same regime.
# If they don't, a figure is being drawn from the wrong predictions.
chk = ScreeningMetrics.summary(y_te, s_te, threshold=thr)
tm  = load_json(cfg.result_path("test_metrics")).get(f"{PRIMARY_FP}|{FIG_REGIME}", {})
print(f"  check  MCC {chk['MCC']:.3f} vs stage3 {tm.get('MCC', float('nan')):.3f} | "
      f"F1 {chk['F1']:.3f} vs {tm.get('F1', float('nan')):.3f} | "
      f"BEDROC {chk['BEDROC']:.3f} vs {tm.get('BEDROC', float('nan')):.3f}")
assert abs(chk["MCC"] - tm.get("MCC", chk["MCC"])) < 1e-6, \
    "figure predictions do not match STAGE 3 test metrics"

# ---------------- leakage summary table ----------------
rows = []
for regime in ("random", "scaffold", "butina"):
    fp_  = cfg.RESULTS_DIR / f"folds_{PRIMARY_FP}_{regime}.json"
    lk_  = cfg.RESULTS_DIR / f"leakage_{PRIMARY_FP}_{regime}.npz"
    if not fp_.exists():
        continue
    fs = load_json(fp_)
    z  = np.load(lk_) if lk_.exists() else None
    rows.append({
        "regime": regime,
        "median_max_tc": float(z["median_max_tc"]) if z is not None else np.nan,
        "frac_tc_ge_0.6": float(z["frac_above_flag"]) if z is not None else np.nan,
        # CV, not held-out: test prevalences differ across regimes (0.239 / 0.133
        # / 0.300) and both BEDROC and EF move with prevalence, so held-out
        # values are not comparable between regimes. Training prevalences are
        # within 4 points, so the CV comparison is fair.
        "CV_BEDROC_model": float(np.mean(fs["RandomForest"]["BEDROC"])),
        "CV_BEDROC_baseline": float(np.mean(fs["NN_Tanimoto_baseline"]["BEDROC"]))
                              if "NN_Tanimoto_baseline" in fs else np.nan,
    })
if rows:
    lk = pd.DataFrame(rows)
    lk["gap_over_baseline"] = lk.CV_BEDROC_model - lk.CV_BEDROC_baseline
    print("\n  LEAKAGE SUMMARY (cross-validation)")
    print(lk.round(3).to_string(index=False))
    if {"random", "butina"} <= set(lk.regime):
        a = lk.loc[lk.regime == "random", "CV_BEDROC_model"].iloc[0]
        b = lk.loc[lk.regime == "butina", "CV_BEDROC_model"].iloc[0]
        print(f"  leakage estimate (random - butina): {a - b:+.3f}")
    lk.to_csv(cfg.RESULTS_DIR / f"leakage_summary_{PRIMARY_FP}.csv", index=False)

# ---------------- screening ----------------
if cfg.HITS_PATH.exists() and cfg.FINAL_DOCKING_LIST.exists():
    F.fig_applicability_domain(pd.read_csv(cfg.HITS_PATH),
                               pd.read_csv(cfg.FINAL_DOCKING_LIST))

# ---------------- after docking ----------------
# ctrl = pd.read_csv(cfg.TARGET_DIR / f"{cfg.PROTEIN_ID}_{cfg.DOMAIN}_controls_indomain.csv")
# F.fig_docking_control(selected_vina_scores, control_vina_scores)

print("\nfigures written:")
for p in sorted(cfg.FIGURE_DIR.glob("*.png")):
    print("  ", p.name)


# 2. Virtual Screening COCONUT db



> **TODO** From coconut sdf to csv then csv to filtering all done properly

## 2.1 Background Code

In [ ]:
class VirtualScreener:
    """
    Screens the natural product library and builds the docking shortlist.

    Changes from the thesis version:
      - NO decision threshold. The old screen_coconut() kept `probs >= threshold`.
        A probability calibrated on a ~30%-active training prior is meaningless on
        a library whose true active rate is ~0.01%, so the model output is used
        purely as a RANKING.
      - multi-model scoring plus parallel selection, which was the best-performing
        of five rank-fusion algorithms in published comparisons
      - applicability domain computed and REPORTED in bands, not silently applied
      - PAINS/Brenk/Ro5 as flags; only physically motivated docking limits filter
      - a random control set, so you can show the ML prefilter did something
    """

    def __init__(self, cfg, models, train_smiles, train_labels, verbose=True):
        """models: {fp_name: ModelBundle}. Load bundles from disk; nothing else needed."""
        """models: {fp_name: fitted BioactivityModel}. One per representation carried forward."""
        self.cfg = cfg
        self.models = models
        self.train_smiles = np.asarray(train_smiles)
        self.train_labels = np.asarray(train_labels)
        self.verbose = verbose
        self._ref = self._ref_act = None
        self.overlap_ = None

    def _refs(self):
        """Binary ECFP4 reference sets for the applicability domain, built once."""
        if self._ref is None:
            b, ok = SimilarityUtils.to_bits(self.train_smiles)
            self._ref = b
            self._ref_act = b[self.train_labels[ok] == 1]
            if self.verbose:
                print(f"  AD reference: {b.shape[0]} train compounds, {self._ref_act.shape[0]} actives")
        return self._ref, self._ref_act

    def screen(self, input_csv=None, smiles_col="canonical_smiles",
               chunksize=25000, keep_top=5000, standardize=False, nrows=None, ad_min_tc=None):
        """
        Streams the library, scores with every model, and keeps a running union of
        the top `keep_top` per model. Memory stays bounded regardless of library size.

        standardize=False by default: run ChemStandardizer over COCONUT ONCE offline
        and save the result, rather than re-canonicalising tautomers on every pass.
        ad_min_tc: applied BEFORE top-K retention. Score and similarity are
        nearly independent, so gating afterwards threw away 97.7% of what was
        retained (114 of 5,000 survived). Gating inside the pass means keep_top
        returns keep_top in-domain compounds.
        """
        input_csv = input_csv or self.cfg.INPUT_CSV
        ref, _ = (self._refs() if ad_min_tc is not None else (None, None))
        buf, n_seen, n_ok, n_ad_drop = [], 0, 0, 0

        
        for chunk in pd.read_csv(input_csv, chunksize=chunksize,
                                 nrows=nrows, low_memory=False):
            n_seen += len(chunk)
            chunk = chunk.dropna(subset=[smiles_col]).reset_index(drop=True)
            scol = smiles_col
            if standardize:
                chunk = ChemStandardizer.standardize_frame(chunk, smiles_col, verbose=False)
                scol = "std_smiles"
            if len(chunk) == 0:
                continue

            if ad_min_tc is not None:                        #.. AD gate INSIDE the pass
                b, okb = SimilarityUtils.to_bits(chunk[scol])
                tc = np.zeros(len(chunk), dtype=np.float32)
                tc[okb] = SimilarityUtils.max_tanimoto(b, ref)
                keep = tc >= ad_min_tc
                n_ad_drop += int((~keep).sum())
                chunk = chunk.loc[keep].reset_index(drop=True)
                chunk["max_tc_train"] = tc[keep]             #.. carry it; don't recompute later
                if len(chunk) == 0:
                    continue

            scores, mask_all = {}, None
            for fp_name, bm in self.models.items():
                X, ok = bm.featurize(chunk[scol])
                s = np.full(len(chunk), np.nan)
                if X.shape[0]:
                    s[ok] = bm.predict_proba(X)
                scores[f"score_{fp_name}"] = s
                mask_all = ok if mask_all is None else (mask_all & ok)

            sub = chunk.loc[mask_all].copy()
            for k, v in scores.items():
                sub[k] = v[mask_all]
            n_ok += len(sub)
            buf.append(sub)

            df = pd.concat(buf, ignore_index=True)
            keep_mask = np.zeros(len(df), dtype=bool)
            for fp_name in self.models:
                keep_mask |= df[f"score_{fp_name}"].rank(ascending=False,
                                                         method="first").values <= keep_top
            buf = [df.loc[keep_mask].reset_index(drop=True)]
            if self.verbose:
                print(f"    seen {n_seen} | in-domain {n_ok} | retained {len(buf[0])}", end="\r")

        out = (buf[0] if buf else pd.DataFrame()).reset_index(drop=True)
        out.attrs.update(n_library=n_seen, n_scored=n_ok, n_ad_dropped=n_ad_drop,
                         ad_min_tc=ad_min_tc, smiles_col=("std_smiles" if standardize else smiles_col))
        if self.verbose:
            print(f"\n  library {n_seen} | AD-dropped {n_ad_drop} | scored {n_ok} | retained {len(out)}")
        return out
    # ---------------- applicability domain ----------------

    def derive_ad_threshold(self, percentile=5, verbose=True):
        """
        Data-driven applicability domain cutoff.

        Uses ONLY the training structures. No model, no labels, no refitting.
        For each training compound, find its max Tanimoto to any OTHER training
        compound; the low percentile of that distribution is how isolated the
        most peripheral training compound is. A library molecule further from
        the training set than that is outside the space the model learned.

        Replaces a hand-picked 0.3 with a number defensible in methods.
        Takes no dataframe: it is a property of the training set alone.
        """
        ref, _ = self._refs()
        S = SimilarityUtils.tanimoto_matrix(ref, ref)
        np.fill_diagonal(S, -1.0)                       # exclude self-similarity
        nn = S.max(axis=1)
        thr = float(np.percentile(nn, percentile))
        if verbose:
            print(f"  training-set nearest-neighbour Tc: "
                  f"median {np.median(nn):.3f} | "
                  f"25th {np.percentile(nn, 25):.3f} | "
                  f"10th {np.percentile(nn, 10):.3f} | "
                  f"{percentile}th {thr:.3f}")
            print(f"  -> derived AD threshold = {thr:.3f}")
        return thr

    def library_coverage(self, input_csv=None, smiles_col="canonical_smiles",
                         thresholds=(0.2, 0.25, 0.3, 0.35, 0.4, 0.5),
                         chunksize=50000, nrows=None, verbose=True):
        """
        Fraction of the screening library inside the applicability domain at
        each candidate threshold. One streaming pass, no model involved.

        This table belongs in the results: it quantifies the transferability gap
        between ChEMBL synthetic chemistry and natural-product space.

        nrows: cap for a fast estimate; omit for the full library.
        """
        ref, _ = self._refs()
        counts = {t: 0 for t in thresholds}
        n_total = 0
        reader = pd.read_csv(input_csv or self.cfg.INPUT_CSV,
                             chunksize=chunksize, nrows=nrows, low_memory=False)
        for chunk in reader:
            smi = chunk[smiles_col].dropna()
            if not len(smi):
                continue
            b, ok = SimilarityUtils.to_bits(smi)
            if b.shape[0] == 0:
                continue
            tc = SimilarityUtils.max_tanimoto(b, ref)
            n_total += len(tc)
            for t in thresholds:
                counts[t] += int((tc >= t).sum())
            if verbose:
                print(f"    scanned {n_total}", end="\r")
        if verbose:
            print(f"\n  library coverage ({n_total:,} molecules"
                  f"{' sampled' if nrows else ''}):")
            for t in thresholds:
                print(f"    Tc >= {t:<5}: {counts[t]:>8,}  "
                      f"({100 * counts[t] / max(n_total, 1):5.2f}%)")
        return {"n_total": n_total, "counts": counts}

    def add_applicability_domain(self, df, smiles_col=None):
        """
        Adds max_tc_train, max_tc_active and an ad_band label.

        AD membership and novelty pull in opposite directions: screening
        campaigns routinely discard hits above Tc 0.35 to known ligands
        precisely because they are not novel. Reporting bands rather than one
        cut turns that tension into a results table.
        """
        smiles_col = smiles_col or df.attrs.get("smiles_col", "canonical_smiles")
        ref, ref_act = self._refs()
        b, ok = SimilarityUtils.to_bits(df[smiles_col])
        out = df.loc[ok].reset_index(drop=True).copy()
        if "max_tc_train" not in out.columns:          #.. screen() may already carry it
            out["max_tc_train"] = SimilarityUtils.max_tanimoto(b, ref)
        out["max_tc_active"] = SimilarityUtils.max_tanimoto(b, ref_act)
        lo, hi = self.cfg.AD_TC_BANDS
        out["ad_band"] = np.where(out.max_tc_train >= hi, "in_domain_similar",
                          np.where(out.max_tc_train >= lo, "in_domain_novel",
                                   "extrapolation"))
        if self.verbose:
            print("  applicability domain:\n" + out.ad_band.value_counts().to_string())
        out.attrs.update(df.attrs)
        return out
    def add_docking_feasibility(self, df, max_mw=700, max_rotb=10, smiles_col=None):
        """
        Physically motivated limits ONLY. Vina degrades past roughly 10 rotatable
        bonds, and very large ligands do not fit the acetyl-lysine pocket.
        PAINS/Brenk are attached as flags and never used to drop rows.
        """
        smiles_col = smiles_col or df.attrs.get("smiles_col", "canonical_smiles")
        d = df[smiles_col].apply(ChemUtils.calculate_descriptors)
        out = pd.concat([df.reset_index(drop=True), d.reset_index(drop=True)], axis=1)
        out["dockable"] = (out.MW <= max_mw) & (out.RotBonds <= max_rotb)
        out = ChemUtils.annotate_alerts(out, smiles_col)
        if self.verbose:
            print(f"  dockable (MW<={max_mw}, RotB<={max_rotb}): {int(out.dockable.sum())}/{len(out)}")
            print(f"  PAINS flagged {int((out.pains_status=='FLAG').sum())} | "
                  f"Brenk flagged {int((out.brenk_status=='FLAG').sum())}  (flags only, nothing dropped)")
        out.attrs.update(df.attrs)
        return out

    def parallel_selection(self, df, n_final=None, per_model_n=50,
                           require_dockable=True, require_in_domain=True, id_col=None):
        """
        Takes top `per_model_n` from each model, deduplicates, then tops up in
        round-robin until exactly n_final compounds are chosen.

        This avoids inventing an arbitrary "if overlap exceeds X% then expand"
        rule, which has no published basis. Fix n_final first, top up
        deterministically.

        The pairwise Jaccard overlap is a RESULT: consensus only improves
        enrichment when the individual methods are distinctive. Low overlap means
        the fusion is adding information; overlap above ~0.5 means your
        representations are redundant and you should say so.
        """
        n_final = n_final or self.cfg.N_DOCK
        pool = df.copy()
        if require_dockable and "dockable" in pool:
            pool = pool[pool.dockable]
        if require_in_domain and "ad_band" in pool:
            pool = pool[pool.ad_band != "extrapolation"]
        pool = pool.reset_index(drop=True)

        key = id_col if (id_col and id_col in pool) else pool.attrs.get("smiles_col", "canonical_smiles")
        if key not in pool:
            key = [c for c in pool.columns if "smiles" in c.lower()][0]

        fps = list(self.models)
        ranked = {fp: pool.sort_values(f"score_{fp}", ascending=False)[key].tolist() for fp in fps}
        tops = {fp: set(v[:per_model_n]) for fp, v in ranked.items()}

        ov = pd.DataFrame(index=fps, columns=fps, dtype=float)
        for a in fps:
            for b in fps:
                u = len(tops[a] | tops[b])
                ov.loc[a, b] = len(tops[a] & tops[b]) / u if u else np.nan
        self.overlap_ = ov

        chosen, seen = [], set()
        for fp in fps:                                   # pass 1: top-N from each
            for k in ranked[fp][:per_model_n]:
                if k not in seen:
                    seen.add(k); chosen.append(k)
        ptr = {fp: per_model_n for fp in fps}
        while len(chosen) < n_final:                     # pass 2: round-robin top-up
            progressed = False
            for fp in fps:
                if len(chosen) >= n_final:
                    break
                while ptr[fp] < len(ranked[fp]):
                    k = ranked[fp][ptr[fp]]; ptr[fp] += 1
                    progressed = True
                    if k not in seen:
                        seen.add(k); chosen.append(k); break
            if not progressed:
                break

        out = pool[pool[key].isin(chosen)].copy()
        out["consensus_rank"] = out[[f"score_{f}" for f in fps]].rank(ascending=False).mean(axis=1)
        out = out.sort_values("consensus_rank").reset_index(drop=True)

        if self.verbose:
            print(f"  pairwise Jaccard overlap of top-{per_model_n} lists:")
            print(ov.round(3).to_string())
            print(f"  selected {len(out)} (target {n_final}) from a pool of {len(pool)}")
            if "ad_band" in out:
                print("  AD bands of the selection:\n" + out.ad_band.value_counts().to_string())
            if "max_tc_active" in out:
                print(f"  max Tc to training actives: median {out.max_tc_active.median():.3f}, "
                      f"max {out.max_tc_active.max():.3f}")
                print("    (right-shifted vs the library background means your hits are "
                      "analogues of known chemotypes, not novel ones - report the comparison)")
        out.attrs.update(df.attrs)
        return out

    def sample_controls(self, input_csv=None, n=200, smiles_col="canonical_smiles",
                        seed=42, chunksize=25000):
        """
        Uniform random sample of the library by reservoir sampling, so it works in
        one streaming pass without loading 695k rows.

        Dock these under the identical protocol alongside your 150. Comparing the
        two score distributions is the only direct evidence that the ML prefilter
        added anything. If the distributions overlap, that is a real and reportable
        negative result about fingerprint QSAR transferring to natural products.
        """
        input_csv = input_csv or self.cfg.INPUT_CSV
        rng = np.random.default_rng(seed)
        res, k = [], 0
        for chunk in pd.read_csv(input_csv, chunksize=chunksize, low_memory=False):
            chunk = chunk.dropna(subset=[smiles_col])
            for _, row in chunk.iterrows():
                k += 1
                if len(res) < n:
                    res.append(row)
                else:
                    j = rng.integers(0, k)
                    if j < n:
                        res[j] = row
        out = pd.DataFrame(res).reset_index(drop=True)
        out.attrs["smiles_col"] = smiles_col
        if self.verbose:
            print(f"  random control set: {len(out)} of {k} library rows (reservoir, seed={seed})")
        return out

## 2.2 Filtering, Analysing: COCONUT sdf to filtered csv

In [ ]:
# use coconut_filtering.ipynb file to do this. Its once in a while only work....

# save to coconut_pains_lipinski_passed.csv

# OR ||||||
def build_ad_library(cfg, bundles, ad_min_tc=0.30, out_name=None,
                     smiles_col="canonical_smiles", np_likeness_min=0.0,
                     min_annotation=None, chunksize=50000, verbose=True):
    """
    One-time AD pre-filter of the screening library.

    Accepts a ModelBundle OR a {name: ModelBundle} dict; all bundles are trained
    on the same curated set, so the first one's training structures define the
    reference space.

    Prints and returns the full funnel, because those counts are results:
    total -> parseable -> NP-like -> in-domain.
    """
    if isinstance(bundles, dict):
        bundle = next(iter(bundles.values()))
    else:
        bundle = bundles

    out_name = out_name or f"coconut_ad{int(round(ad_min_tc*100)):03d}.csv"
    out_path = cfg.BASE_DIR / "coconut" / out_name
    refbits, _ = SimilarityUtils.to_bits(bundle.train_smiles)
    if verbose:
        print(f"  reference: {refbits.shape[0]} training compounds ({bundle.fp_name} bundle)")
        print(f"  AD threshold: Tc >= {ad_min_tc}")
        print(f"  NP-likeness min: {np_likeness_min} | annotation min: {min_annotation}")

    kept, n_total, n_smiles, n_np, n_annot = [], 0, 0, 0, 0
    for chunk in pd.read_csv(cfg.COCONUT_CSV, chunksize=chunksize, low_memory=False):
        n_total += len(chunk)
        chunk = chunk.dropna(subset=[smiles_col]); n_smiles += len(chunk)
        if np_likeness_min is not None:
            col = next((c for c in chunk.columns if "likeness" in c.lower()), None)
            if col:
                chunk = chunk[pd.to_numeric(chunk[col], errors="coerce") > np_likeness_min]
        n_np += len(chunk)
        if min_annotation is not None:
            col = next((c for c in chunk.columns if "annotation" in c.lower()), None)
            if col:
                chunk = chunk[pd.to_numeric(chunk[col], errors="coerce") >= min_annotation]
        n_annot += len(chunk)
        if not len(chunk):
            continue
        b, ok = SimilarityUtils.to_bits(chunk[smiles_col])
        tc = np.zeros(len(chunk), dtype=np.float32)
        tc[ok] = SimilarityUtils.max_tanimoto(b, refbits)
        sel = chunk.loc[tc >= ad_min_tc].copy()
        sel["max_tc_train"] = tc[tc >= ad_min_tc]
        kept.append(sel)
        if verbose:
            print(f"    {n_total:,} scanned | {sum(len(k) for k in kept):,} in-domain", end="\r")

    out = pd.concat(kept, ignore_index=True) if kept else pd.DataFrame()
    out.to_csv(out_path, index=False)
    funnel = {"total": n_total, "with_smiles": n_smiles, "np_like": n_np,
              "annotated": n_annot, "in_domain": len(out), "ad_min_tc": ad_min_tc}
    if verbose:
        print(f"\n  FUNNEL: {n_total:,} -> smiles {n_smiles:,} -> NP-like {n_np:,} "
              f"-> annotated {n_annot:,} -> in-domain (Tc>={ad_min_tc}) {len(out):,}")
        print(f"  saved -> {out_path}")
    save_json(funnel, cfg.result_path("library_funnel"))
    return out_path, funnel



## 2.3 Screening

In [ ]:


# ============================================================
# STAGE 6a  BUILD AD-FILTERED LIBRARY        run ONCE per target
# reads: coconut csv + one bundle    writes: coconut_ad030.csv
# ============================================================
AD_MIN_TC     = 0.30       #.. from the coverage table: ~8.3% of the library, ~61k compounds
NP_LIKENESS   = 0.0        #.. COCONUT contains synthetics (Hoechst 33342, MLS compounds)
MIN_ANNOTATION = None      #.. set to 2 if the annotation column exists and you want stricter

bundles = {fp: ModelBundle.load(cfg.bundle_path(fp)) for fp in TOP_FPS}

AD_LIB_PATH, funnel = build_ad_library(
    cfg, bundles, ad_min_tc=AD_MIN_TC,
    np_likeness_min=NP_LIKENESS, min_annotation=MIN_ANNOTATION)

cfg.INPUT_CSV = AD_LIB_PATH        #.. everything downstream now reads the small file




In [ ]:


# ============================================================
# STAGE 6b  SCREEN THE AD LIBRARY
# reads: coconut_ad030.csv + bundles   writes: scored, docking list, 2 control sets
# Runs standalone after a kernel restart; needs cfg and the pkl files only.
# ============================================================
bundles = {fp: ModelBundle.load(cfg.bundle_path(fp)) for fp in TOP_FPS}
cfg.INPUT_CSV = cfg.BASE_DIR / "coconut" / f"coconut_ad{int(round(AD_MIN_TC*100)):03d}.csv"
cfg.AD_TC_BANDS = [AD_MIN_TC, 0.50]          #.. keep bands consistent with the gate

n_rows = sum(1 for _ in open(cfg.INPUT_CSV)) - 1
print(f"  AD library: {cfg.INPUT_CSV.name}, {n_rows:,} rows (Tc >= {AD_MIN_TC})")

vs = VirtualScreener(cfg, bundles,
                     train_smiles=bundles[PRIMARY_FP].train_smiles,
                     train_labels=bundles[PRIMARY_FP].train_labels)

# no keep_top: the AD gate already reduced the library, so score everything and
# select from the complete ranking rather than layering an arbitrary second cut
scored = vs.screen(cfg.INPUT_CSV, smiles_col="canonical_smiles",
                   chunksize=25000, keep_top=10**9, standardize=False)
scored = vs.add_applicability_domain(scored)          # max_tc_train already carried
scored = vs.add_docking_feasibility(scored, max_mw=800, max_rotb=12)
scored.to_csv(cfg.HITS_PATH, index=False)

final = vs.parallel_selection(scored, n_final=cfg.N_DOCK,
                              per_model_n=cfg.PER_MODEL_N,   #.. 60 each, NOT N_DOCK
                              id_col="identifier")
final.to_csv(cfg.FINAL_DOCKING_LIST, index=False)

# ---- TWO control sets ----
# (a) random from the AD library: isolates whether the ML RANKING adds anything
#     beyond the similarity gate. This is the control that matters.
ctrl_ad = vs.add_applicability_domain(
    vs.sample_controls(cfg.INPUT_CSV, n=cfg.N_CONTROLS, seed=cfg.SEED))
ctrl_ad.to_csv(cfg.TARGET_DIR / f"{cfg.PROTEIN_ID}_{cfg.DOMAIN}_controls_indomain.csv", index=False)

# (b) random from the FULL library: shows what an unfiltered screen would give
ctrl_full = vs.add_applicability_domain(
    vs.sample_controls(cfg.COCONUT_CSV, n=cfg.N_CONTROLS, seed=cfg.SEED + 1))
ctrl_full.to_csv(cfg.CONTROLS_PATH, index=False)

# ---- non-circular novelty comparison: three numbers, not two ----
print(f"\n  median Tc(training actives):"
      f"\n    AD library pool     {scored.max_tc_active.median():.3f}  (n={len(scored)})"
      f"\n    ML selection        {final.max_tc_active.median():.3f}  (n={len(final)})"
      f"\n    random, in-domain   {ctrl_ad.max_tc_active.median():.3f}"
      f"\n    random, full library {ctrl_full.max_tc_active.median():.3f}")

save_json({"ad_min_tc": AD_MIN_TC, "n_ad_library": int(n_rows),
           "n_scored": int(len(scored)), "n_dockable": int(scored.dockable.sum()),
           "n_selected": int(len(final)),
           "overlap": vs.overlap_.astype(float).to_dict(),
           "median_tc_active": {
               "pool": float(scored.max_tc_active.median()),
               "selection": float(final.max_tc_active.median()),
               "random_indomain": float(ctrl_ad.max_tc_active.median()),
               "random_full": float(ctrl_full.max_tc_active.median())},
           "pains_flagged_selection": int((final.pains_status == "FLAG").sum()),
           "brenk_flagged_selection": int((final.brenk_status == "FLAG").sum())},
          cfg.result_path("screening"))


In [ ]:
lib = pd.read_csv(cfg.COCONUT_CSV, low_memory=False)
print([c for c in lib.columns if 'lik' in c.lower() or 'annot' in c.lower()])
lib = lib[lib["np_likeness"] > 0]                  #.. column name may differ; check first

In [ ]:
print(len(lib))
print(lib['qed_drug_likeliness'].describe())
lib['annotation_level'].describe()

## 2.3.2 VERIFY DOCKING WITH ANOTHER SWISSMODEL PROPERLY.

In [ ]:
from pathlib import Path
import subprocess


PDBQT = cfg.DOCKING_DIR / "ligands/pdbqt"
OUT_DIR = cfg.DOCKING_DIR / "out/verify"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REC = cfg.DOCKING_DIR / "receptor/swissmodel1.pdbqt"
CONFIG = cfg.DOCKING_DIR / "receptor/swissmodel1_receptor.box.txt"

ligands = sorted(PDBQT.glob("CNP*.pdbqt"))

for lig in ligands:
    log_file = OUT_DIR / f"{lig.stem}.log"
    result = subprocess.run([
        "vina", "--receptor", str(REC), "--ligand", str(lig), "--config", str(CONFIG),
        "--exhaustiveness", "32", "--num_modes", "9", "--seed", "42",
        "--out", str(OUT_DIR / f"{lig.stem}.pdbqt")
    ], capture_output=True, text=True)

    print(result.stdout)              # shows in notebook
    log_file.write_text(result.stdout)  # saves to file

## 2.4 VERIFY ANALYZE OUTPUT RESULTS

In [ ]:
# ============================================================
# VERIFY  -  cross-check every saved artifact. Reads only; changes nothing.
# ============================================================
import numpy as np, pandas as pd, json
from pathlib import Path
R = cfg.RESULTS_DIR
ok_all = True

def chk(label, cond, detail=""):
    global ok_all
    ok_all &= bool(cond)
    print(f"  [{'PASS' if cond else 'FAIL'}] {label}{'  ' + detail if detail else ''}")

print("=== 1. SMOKE contamination ===")
tp = json.loads((R / "tuned_params.json").read_text())
single = all(not isinstance(v, (list, tuple)) or len(v) == 1
             for d in tp.values() for v in d.values())
chk("tuned_params present", len(tp) > 0, f"{list(tp)}")
print(f"       {json.dumps(tp, indent=8)[:400]}")
val = json.loads((R / "validation.json").read_text())
chk("y-scrambling n >= 10", len(val['scrambling']['scrambled']) >= 10,
    f"n={len(val['scrambling']['scrambled'])}")
chk("seed sensitivity n >= 10",
    len(val['seed_sensitivity'][cfg.RANKING_METRIC]) >= 10,
    f"n={len(val['seed_sensitivity'][cfg.RANKING_METRIC])}")

print("\n=== 2. Expected files ===")
for f in ["test_metrics.json", "validation.json", "screening.json",
          "library_funnel.json", "leakage_summary_ecfp4_maccs.csv"]:
    chk(f, (R / f).exists())
for fp, regs in {PRIMARY_FP: ["random","scaffold","butina"],
                 SECONDARY_FP: ["random","scaffold"]}.items():
    for r in regs:
        for pat in ["benchmark_{}_{}.csv", "folds_{}_{}.json",
                    "leakage_{}_{}.npz", "preds_{}_{}.npz"]:
            chk(pat.format(fp, r), (R / pat.format(fp, r)).exists())

print("\n=== 3. preds_*.npz reproduce test_metrics.json ===")
tm = json.loads((R / "test_metrics.json").read_text())
for key in tm:
    fp, reg = key.split("|")
    z = np.load(R / f"preds_{fp}_{reg}.npz")
    s = ScreeningMetrics.summary(z["y_true"], z["y_score"],
                                 threshold=float(z["threshold"]))
    for m in ("MCC", "F1", "BEDROC", "ROC-AUC", "AUC-PR"):
        chk(f"{key} {m}", abs(s[m] - tm[key][m]) < 1e-6,
            f"{s[m]:.4f} vs {tm[key][m]:.4f}")

print("\n=== 4. Leakage monotonicity ===")
lk = pd.read_csv(R / f"leakage_summary_{PRIMARY_FP}.csv")
lk = lk.sort_values("median_max_tc", ascending=False)
print(lk.round(3).to_string(index=False))
chk("CV BEDROC falls with similarity",
    lk.CV_BEDROC_model.is_monotonic_decreasing)
chk("all regimes beat the baseline", (lk.gap_over_baseline > 0).all())
chk("skeleton_overlap == 0 everywhere",
    all(int(np.load(R / f"leakage_{PRIMARY_FP}_{r}.npz")["skeleton_overlap"]) == 0
        for r in ("random","scaffold","butina")))

print("\n=== 5. Bundles ===")
for fp in TOP_FPS:
    b = joblib.load(cfg.bundle_path(fp))
    bench = pd.read_csv(R / f"benchmark_{fp}_scaffold.csv")
    champ = bench[bench.Model != "NN_Tanimoto_baseline"].sort_values("MeanRank").iloc[0]["Model"]
    chk(f"{fp} fp_name", b.fp_name == fp)
    chk(f"{fp} n_train == curated", b.meta["n_train"] == len(pd.read_csv(cfg.CURATED_PATH)),
        f"{b.meta['n_train']}")
    chk(f"{fp} model == benchmark champion",
        b.meta.get("model", type(b.model.named_steps['clf']).__name__).startswith(champ[:6]),
        f"bundle={b.meta.get('model','?')} benchmark={champ}")

print("\n=== 6. Screening ===")
sj = json.loads((R / "screening.json").read_text())
fn = json.loads((R / "library_funnel.json").read_text())
print(f"  funnel: {fn['total']:,} -> NP-like {fn['np_like']:,} -> in-domain {fn['in_domain']:,}")
chk("selected == N_DOCK", sj["n_selected"] == cfg.N_DOCK, str(sj["n_selected"]))
ov = pd.DataFrame(sj["overlap"])
chk("two screening arms", ov.shape[0] == 2, f"shape {ov.shape}")
offdiag = ov.values[~np.eye(len(ov), dtype=bool)]
chk("arms are distinctive (Jaccard < 0.8)", offdiag.max() < 0.8, f"{offdiag.max():.3f}")
mt = sj["median_tc_active"]
chk("selection > in-domain random", mt["selection"] > mt["random_indomain"],
    f"{mt['selection']:.3f} vs {mt['random_indomain']:.3f}")

print("\n" + ("ALL CHECKS PASSED" if ok_all else "SOME CHECKS FAILED - see above"))

# 3. Molecular Docking



> **TODO:** docking steps verify? 



> Docking Steps

> 1. Get the pdb from rcsb

> 2. If it starts from later and not one, swiss model (mine had 1 to 41 just later?)

> 3. Get active residues from the paper. 

> 4. Comparision between original and swiss-modeled pdb file (no JQ1 and another unchained instead)

> 5. Is it same or uniform?

> 6. Remove water, add hydrogen, add kolman charge, non-polar hydrogens were merged

> 7. Mark it on pdb and grid check and center

> 8. Ligand prep (got native ligand JQ1 from ADT 3MXF)

> 9. dock

> 10. Check error, rmsd, orientation, redocking...

> 

## 3.1 Preperation Ligands, Receptor



Here, you will prepare ligands, receptor and stuffs. From Autodocktools or other methods. Either done through here or commands here. 





Do main preparation steps of receptor from autodocktools GUI. For ligands see below:

### 3.1.1 Ligands Prep



First, see the folders, see how many docking ligands are there. 

Then, convert the smiles to sdf

Then, introduce some controls and convert them to sdf too

Then, sdf to pdbqt with meeko

In [ ]:
SDF  = cfg.DOCKING_DIR/"ligands/sdf";   SDF.mkdir(parents=True, exist_ok=True)

PDBQT= cfg.DOCKING_DIR/"ligands/pdbqt"; PDBQT.mkdir(parents=True, exist_ok=True)

OUT  = cfg.DOCKING_DIR/"out";           OUT.mkdir(parents=True, exist_ok=True)



df = pd.read_csv(cfg.TARGET_DIR/"CHEMBL1163125_BD1_final_docking_list.csv")

df = df[["identifier","canonical_smiles","molecular_weight","heavy_atom_count"]].dropna(subset=["canonical_smiles"])

df = df.drop_duplicates("identifier")

print(len(df), "ligands")

df.head()




In [ ]:
df['heavy_atom_count'].describe()

In [ ]:
# convert ligands to sdf



lfc = rdMolStandardize.LargestFragmentChooser()

failed = []



for _, r in df.iterrows():

    name = str(r.identifier)

    m = Chem.MolFromSmiles(r.canonical_smiles)

    if m is None:

        failed.append((name, "parse")); continue

    m = lfc.choose(m)                      # strip salts/counterions

    m = Chem.AddHs(m, addCoords=True)      # Meeko requires explicit Hs

    if AllChem.EmbedMolecule(m, randomSeed=0xf00d, useRandomCoords=True) != 0:

        failed.append((name, "embed")); continue

    try:

        AllChem.MMFFOptimizeMolecule(m, maxIters=500)

    except Exception:

        AllChem.UFFOptimizeMolecule(m, maxIters=500)   # MMFF lacks params for some atoms

    m.SetProp("_Name", name)

    Chem.MolToMolFile(m, str(SDF/f"{name}.sdf"))



print("SDF written:", len(list(SDF.glob('*.sdf'))), "| failed:", failed)

In [ ]:
# control ligands to sdf





controls = {

 # swapped: this string matches the 3MXF crystal ligand

 "CTRL_JQ1_plus":  "CC1=C(C)C2=C(S1)N3C(C)=NN=C3[C@H](CC(=O)OC(C)(C)C)N=C2C1=CC=C(Cl)C=C1",

 "CTRL_JQ1_minus": "CC1=C(C)C2=C(S1)N3C(C)=NN=C3[C@@H](CC(=O)OC(C)(C)C)N=C2C1=CC=C(Cl)C=C1",

 # corrected structures

 "CTRL_IBET762":   "CCNC(=O)C[C@H]1C2=NN=C(N2C3=C(C=C(C=C3)OC)C(=N1)C4=CC=C(C=C4)Cl)C",

 "CTRL_RVX208":    "COC1=CC(OC)=C(C(NC(C2=CC(C)=C(OCCO)C(C)=C2)=N3)=O)C3=C1",

  "CTRL_Caffeine":  "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",

 "CTRL_Aspirin":   "CC(=O)OC1=CC=CC=C1C(=O)O",

}



for name, smi in controls.items():

    m = Chem.AddHs(lfc.choose(Chem.MolFromSmiles(smi)), addCoords=True)

    AllChem.EmbedMolecule(m, randomSeed=0xf00d)

    AllChem.MMFFOptimizeMolecule(m, maxIters=500)

    m.SetProp("_Name", name)

    Chem.MolToMolFile(m, str(SDF/f"{name}.sdf"))

print("total SDF:", len(list(SDF.glob('*.sdf'))))

In [ ]:
# SDF to pdbqt with meeko



bad = []

for f in sorted(SDF.glob("*.sdf")):

    out = PDBQT/f"{f.stem}.pdbqt"

    if out.exists(): out.unlink()          # never trust a stale file



    p = subprocess.run(["mk_prepare_ligand.py","-i",str(f),"-o",str(out)],

                       capture_output=True, text=True)



    if p.returncode != 0 or not out.exists():

        bad.append((f.stem, p.stderr.strip()[:120]))



print("PDBQT:", len(list(PDBQT.glob('*.pdbqt'))), "| failed:", len(bad))

for b in bad[:10]: print(b)

### 3.1.2 Ligands prep coconut sdf to pdbqt

In [ ]:
from openbabel import pybel

final_docking_list = pd.read_csv(cfg.FINAL_DOCKING_LIST)['identifier'].to_list()


for mol in pybel.readfile("sdf", str(cfg.COCONUT_SDF)):
    cid = mol.data["IDENTIFIER"]
    if cid in final_docking_list: 
        mol.write("sdf", f"{cfg.DOCKING_DIR}/ligands/sdf/{cid}.sdf", overwrite=True)

#### MINIMIZE and to PDBQT

In [ ]:
# run in terminal
# for f in *.sdf; do obabel "$f" -O "minimized/$f" --minimize --ff MMFF94 --steps 250; done
# for f in *.sdf; do mk_prepare_ligand.py -i "$f" -o "../pdbqt/${f%.sdf}.pdbqt"; done

## 3.2 ReDocking



Used to check if our docking protocal is valid. Checked with obrms and/or pymol

In [ ]:
REC=cfg.DOCKING_DIR/"receptor"

REC

In [ ]:
%%bash -s "$REC"

REC=$1



mkdir -p out



# --- redock: JQ1 back into 3MXF (validates the docking protocol itself) ---

vina --receptor $REC/3mxf_original_receptor.pdbqt \

     --ligand   $REC/jq1_native.pdbqt \

     --config   $REC/3mxf_original_receptor_box.txt \

     --exhaustiveness 32 --num_modes 9 --seed 42 \

     --out out/redock_crystal.pdbqt



# --- redock: 14Z back into the homology model (validates the model's own pocket) ---

vina --receptor $REC/brd4_swiss_receptor.pdbqt \

     --ligand   $REC/14z_native.pdbqt \

     --config   $REC/brd4_swiss_receptor_box.txt \

     --exhaustiveness 32 --num_modes 9 --seed 42 \

     --out out/redock_homology.pdbqt





mk_export.py out/redock_crystal.pdbqt  -s out/redock_crystal.sdf

mk_export.py out/redock_homology.pdbqt -s out/redock_homology.sdf

mk_export.py out/cross_model.pdbqt     -s out/cross_model.sdf



echo "--- redock crystal (JQ1 -> 3MXF) ---";     obrms -f $REC/jq1_native.sdf  out/redock_crystal.sdf  | head -3

echo "--- redock homology (14Z -> model) ---";    obrms -f $REC/14z_native.sdf  out/redock_homology.sdf | head -3


## 3.3 Main Docking

Beofre starting here, make sure you have prepared the ligands and receptors

In [ ]:

import subprocess
from pathlib import Path

REC  = cfg.DOCKING_DIR/"receptor/rec_prod.pdbqt"
CEN  = ("28.579","15.975","-2.404")
SIZE = "18"
PDBQT, OUT = cfg.DOCKING_DIR/Path("ligands/pdbqt"), cfg.DOCKING_DIR/Path("out")
OUT.mkdir(exist_ok=True)

files = sorted(PDBQT.glob("*.pdbqt"))
print(len(files), "ligands")

# for i, f in enumerate(files, 1):
#     n = f.stem
#     op, lg = OUT/f"{n}.pdbqt", OUT/f"{n}.log"
#     if op.exists(): op.unlink()
#     cmd = ["vina","--receptor",REC,"--ligand",str(f),
#            "--center_x",CEN[0],"--center_y",CEN[1],"--center_z",CEN[2],
#            "--size_x",SIZE,"--size_y",SIZE,"--size_z",SIZE,
#            "--exhaustiveness","32","--num_modes","9","--seed","42",
#            "--cpu","8","--out",str(op)]
#     with open(lg,"w") as L:
#         subprocess.run(cmd, stdout=L, stderr=subprocess.STDOUT)
#     if i % 20 == 0: print(f"{i}/{len(files)}")

# print("done:", len(list(OUT.glob('CNP*.pdbqt'))) + len(list(OUT.glob('CTRL*.pdbqt'))))

```
pymol -c data/brd4/docking/build_complexes.pml
plip -f CNP0072228_complex.pdb -t 
for f in *_complex.pdb; do plip -f "$f" -t; done
```


```
admet_predict \
    --data_path data/brd4/out/results/final_docked_ranked_metadata.csv \
    --save_path data/brd4/out/results/CHEMBL1163125_BD1_admetai_preds.csv \
    --smiles_column canonical_smiles

```

Files: 
data/brd4/out/results/CHEMBL1163125_BD1_admetai_preds.csv
data/brd4/out/results/final_docked_ranked_metadata.csv

In [ ]:
import pandas as pd

rows = []
for f in sorted(OUT.glob("*.pdbqt")):
    if f.stem in ("val", "cross"):
        continue
    line = next((l for l in open(f) if "VINA RESULT" in l), None)
    if line:
        ha = Chem.MolFromMolFile(
            f"{cfg.DOCKING_DIR}/ligands/sdf/{f.stem}.sdf"
        ).GetNumHeavyAtoms()
        s = float(line.split()[3])
        rows.append({"identifier": f.stem, "score": s, "HA": ha, "LE": -s / ha})

res = pd.DataFrame(rows).sort_values("score").reset_index(drop=True)

# Read metadata CSV
df_meta = pd.read_csv(cfg.FINAL_DOCKING_LIST)

# Inner join retains ONLY molecules present in docking output (~100)
res_merged = df_meta.merge(
    res[["identifier", "score", "LE"]], on="identifier", how="inner"
).sort_values("score").reset_index(drop=True)

# Assign rank based strictly on completed docking scores
res_merged["rank"] = res_merged.index + 1

# Save merged dataset
res_merged.to_csv(f"{cfg.RESULTS_DIR}/final_docked_ranked_metadata.csv", index=False)

print(res_merged.head(100).to_string(index=False))
print("\n--- controls ---")
print(
    res_merged[res_merged.identifier.str.startswith("CTRL_")].to_string(
        index=False
    )
)
print(
    "\nscore~HA corr:",
    round(
        res_merged[~res_merged.identifier.str.startswith("CTRL_")][
            ["score", "heavy_atom_count"]
        ]
        .corr()
        .iloc[0, 1],
        3,
    ),
)

## 3.4 KNOWN MEDICINE:
Check against known BD1 inhibitors
1.  Amredobresib (Synonyms: BI894999):
   ```
   CNC1=NC(C2=NC3=C(N2CC4=CC=CC=C4)C=C(N5CCN(CC5)C)C=C3)=CN6C(C)=NN=C16
   ```

2. Birabresib (OTX015)
   ```
   CC1=C(SC2=C1C(=NC(C3=NN=C(N32)C)CC(=O)NC4=CC=C(C=C4)O)C5=CC=C(C=C5)Cl)C
   ```


In [ ]:
# control ligands to sdf

lfc = rdMolStandardize.LargestFragmentChooser()

SDF  = cfg.DOCKING_DIR/"ligands/sdf/controls";   SDF.mkdir(parents=True, exist_ok=True)

PDBQT= cfg.DOCKING_DIR/"ligands/pdbqt/controls"; PDBQT.mkdir(parents=True, exist_ok=True)

OUT  = cfg.DOCKING_DIR/"out/controls";           OUT.mkdir(parents=True, exist_ok=True)




controls = {

 # swapped: this string matches the 3MXF crystal ligand

 "CTRL_JQ1_plus":  "CC1=C(C)C2=C(S1)N3C(C)=NN=C3[C@H](CC(=O)OC(C)(C)C)N=C2C1=CC=C(Cl)C=C1",


 # corrected structures

 "CTRL_IBET762":   "CCNC(=O)C[C@H]1C2=NN=C(N2C3=C(C=C(C=C3)OC)C(=N1)C4=CC=C(C=C4)Cl)C",

 "CTRL_RVX208":    "COC1=CC(OC)=C(C(NC(C2=CC(C)=C(OCCO)C(C)=C2)=N3)=O)C3=C1",

  "CTRL_Caffeine":  "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",

 "CTRL_Aspirin":   "CC(=O)OC1=CC=CC=C1C(=O)O",
 "CTRL_Amredobresib":"CNC1=NC(C2=NC3=C(N2CC4=CC=CC=C4)C=C(N5CCN(CC5)C)C=C3)=CN6C(C)=NN=C16",
 "CTRL_Birabresib": "CC1=C(SC2=C1C(=NC(C3=NN=C(N32)C)CC(=O)NC4=CC=C(C=C4)O)C5=CC=C(C=C5)Cl)C"

}



for name, smi in controls.items():

    m = Chem.AddHs(lfc.choose(Chem.MolFromSmiles(smi)))
    cid = AllChem.EmbedMolecule(m, randomSeed=0xf00d)
    if cid == -1:
        print(f"{name}: embedding failed"); continue
    AllChem.MMFFOptimizeMolecule(m, maxIters=500)

    m.SetProp("_Name", name)

    Chem.MolToMolFile(m, str(SDF/f"{name}.sdf"))

print("total SDF:", len(list(SDF.glob('*.sdf'))))


# SDF to pdbqt with meeko



bad = []

for f in sorted(SDF.glob("*.sdf")):

    out = PDBQT/f"{f.stem}.pdbqt"

    if out.exists(): out.unlink()          # never trust a stale file



    p = subprocess.run(["mk_prepare_ligand.py","-i",str(f),"-o",str(out)],

                       capture_output=True, text=True)



    if p.returncode != 0 or not out.exists():

        bad.append((f.stem, p.stderr.strip()[:120]))



print("PDBQT:", len(list(PDBQT.glob('*.pdbqt'))), "| failed:", len(bad))

for b in bad[:10]: print(b)

In [ ]:
from pathlib import Path
import subprocess


PDBQT = cfg.DOCKING_DIR / "ligands/pdbqt/controls"
OUT_DIR = cfg.DOCKING_DIR / "out/controls"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REC = cfg.DOCKING_DIR / "receptor/swissmodel1.pdbqt"
CONFIG = cfg.DOCKING_DIR / "receptor/swissmodel1_receptor.box.txt"

ligands = sorted(PDBQT.glob("CTRL*.pdbqt"))

for lig in ligands:
    log_file = OUT_DIR / f"{lig.stem}.log"
    result = subprocess.run([
        "vina", "--receptor", str(REC), "--ligand", str(lig), "--config", str(CONFIG),
        "--exhaustiveness", "32", "--num_modes", "9", "--seed", "42", 
        "--out", str(OUT_DIR / f"{lig.stem}.pdbqt")
    ], capture_output=True, text=True)

    print(result.stdout)              # shows in notebook
    log_file.write_text(result.stdout)  # saves to file

<!-- -->